In [ ]:
import csv
import os
import re
import pickle
import warnings
import numpy as np
import pandas as pd
import scipy
import sklearn
import ukbiobank
import ukbiobank.utils.utils
from ukbiobank.utils import loadCsv
from ukbiobank.utils import addFields
from ukbiobank.utils.utils import fieldIdsToNames
from sklearn.preprocessing import StandardScaler

In [ ]:
# Upload UK Bioabank csv
csv_path = '/UK_BB/ukbbdataukb/ukb.csv'
ukb = ukbiobank.ukbio(ukb_csv=csv_path)

In [ ]:
# Create directories & save
base_path = f'/UK_BB/brainbody'
lifestyle_path = os.path.join(base_path, 'lifestyle')
data_path = os.path.join(lifestyle_path, 'data')
os.makedirs(base_path, exist_ok=True)
os.makedirs(lifestyle_path, exist_ok=True)
os.makedirs(data_path, exist_ok=True)

# Electronic device use

In [ ]:
df_device_use = ukbiobank.utils.utils.loadCsv(ukbio=ukb, fields=['eid',
1110, #Length of mobile phone use
1120, #Weekly usage of mobile phone in last 3 months
1130, #Hands-free device/speakerphone use with mobile phone in last 3 month
1140, #Difference in mobile phone use compared to two years previously
1150, #Usual side of head for mobile phone use
2237, #Plays computer games
])
df_device_use_names = ukbiobank.utils.utils.fieldIdsToNames(ukbio=ukb, df=df_device_use)
df_device_use_names.to_csv(os.path.join(data_path, 'electronic_device_use_vars.csv'), index=False)

In [ ]:
# Upload variables directly from csv if ukbb utils do not work
upload_columns = [
'eid',
'1', #a
'2', #b
'3', #c
]

file_path = '/UK_BB/ukbbdataukb/ukb.csv'
with open(file_path, 'r') as f:
    headers = f.readline().strip().split(',')
    headers = [header.strip('"') for header in headers]

missing_columns = [col for col in upload_columns if not any(header.startswith(col) for header in headers)]
if missing_columns:
    raise ValueError(f"The following required columns are missing: {missing_columns}")
else:
    print("All required columns are present. Proceeding with loading the data.")

    df = pd.read_csv(file_path, usecols=lambda col: (col == 'eid' or any(col.startswith(prefix) for prefix in upload_columns[1:])))
    df.to_csv(os.path.join(data_path, 'df.csv'), index=False)

In [ ]:
print('% missing')
with pd.option_context('display.max_rows', None):
    display(((df_device_use_names.isna().sum() / len(df_device_use_names)).round(2)*100).sort_values(ascending=False))

In [ ]:
# Filter Instance 2
df_device_use_names = pd.read_csv(os.path.join(data_path, 'electronic_device_use_vars.csv'))
device_use_i2 = df_device_use_names[['eid'] + df_device_use_names.filter(regex=r'2\.\d$').columns.tolist()]
device_use_i2 = device_use_i2.dropna(axis=0).reset_index(drop=True)
device_use_i2.columns = device_use_i2.columns.str.replace('-2.0', '')
device_use_i2.to_csv(os.path.join(data_path, 'electronic_device_use_i2_raw.csv'), index=False)
with pd.option_context('display.max_rows', None):
    display(((device_use_i2.isna().sum() / len(device_use_i2)).round(2)*100).sort_values(ascending=False))

In [ ]:
# Count NA and negative values
print('Shape', device_use_i2.shape)
print('NA\n', (device_use_i2.isna()).sum().sort_values(ascending=False))
print('Number of -1 responses\n', (device_use_i2 == -1).sum().sort_values(ascending=False))
print('Number of -3 responses\n', (device_use_i2 == -3).sum().sort_values(ascending=False))

In [ ]:
# Demmy-encode 'Difference in mobile phone use compared to two years previously'
device_use_i2_encode = device_use_i2.copy()

phone_use_mapping = {
0:	'No',
1:	'Yes, use is now less frequent',
2:	'Yes, use is now more frequent',
3:	"I didn't use a mobile phone two years ago",
-1: 'Do not know / Prefer not to answer',
-3: 'Do not know / Prefer not to answer'
}

# Convert to numeric
device_use_i2_encode['Difference in mobile phone use compared to two years previously'] = (
    device_use_i2_encode['Difference in mobile phone use compared to two years previously']
    .apply(pd.to_numeric, errors='raise')  # FAIL if non-numeric values exist
    .astype('Int64')  # Will raise if NaN (Int64 can't handle NaN without permission)
)

# Validate values before mapping
valid_values = {0, 1, 2, 3, -1, -3}
invalid_mask = ~device_use_i2_encode['Difference in mobile phone use compared to two years previously'].isin(valid_values)

if invalid_mask.any():
    invalid_values = device_use_i2_encode.loc[invalid_mask, 'Difference in mobile phone use compared to two years previously'].unique()
    raise ValueError(f"Unexpected values found: {invalid_values}. Allowed: {valid_values}")

# Map to labels
device_use_i2_encode['Difference in mobile phone use compared to two years previously'] = (
    device_use_i2_encode['Difference in mobile phone use compared to two years previously']
    .map(phone_use_mapping)  # No fillna(), will raise if unexpected values exist
)

# One-hot encode
dummies = pd.get_dummies(
    device_use_i2_encode['Difference in mobile phone use compared to two years previously'],
    prefix='Difference in mobile phone use compared to two years previously',
    prefix_sep=' - '
)

# Drop original column & concat
device_use_i2_encode_dummy = pd.concat([
    device_use_i2_encode.drop(columns='Difference in mobile phone use compared to two years previously'),
    dummies
], axis=1)


# Validate the resulting dataframe
print('NA\n', (device_use_i2_encode_dummy.isna()).sum().sort_values(ascending=False))
print('MIN\n', device_use_i2_encode_dummy.min())
print('NEG\n', (device_use_i2_encode_dummy < 0).sum().sort_values(ascending=False))

dummies.columns.tolist()

In [ ]:
# Dummy-encode 'Usual side of head for mobile phone use'
phone_side_mapping = {
    1: 'Left',
    2: 'Right',
    3: 'Equally left and right',
    -1: 'Do not know / Prefer not to answer',
    -3: 'Do not know / Prefer not to answer'
}

# Convert to numeric
device_use_i2_encode_dummy['Usual side of head for mobile phone use'] = (
    device_use_i2_encode_dummy['Usual side of head for mobile phone use']
    .apply(pd.to_numeric, errors='raise')
    .astype('Int64')
)

# Validate values before mapping
valid_values = {1, 2, 3, -1, -3}
invalid_mask = ~device_use_i2_encode_dummy['Usual side of head for mobile phone use'].isin(valid_values)

if invalid_mask.any():
    invalid_values = device_use_i2_encode_dummy.loc[invalid_mask, 'Usual side of head for mobile phone use'].unique()
    raise ValueError(f"Unexpected values found: {invalid_values}. Allowed: {valid_values}")

# Map to labels
device_use_i2_encode_dummy['Usual side of head for mobile phone use'] = (
    device_use_i2_encode_dummy['Usual side of head for mobile phone use']
    .map(phone_side_mapping)
)

# One-hot encode
dummies = pd.get_dummies(
    device_use_i2_encode_dummy['Usual side of head for mobile phone use'],
    prefix='Usual side of head for mobile phone use',
    prefix_sep=' - '
)

# Drop original column & concat
device_use_i2_encode_dummy = pd.concat([
    device_use_i2_encode_dummy.drop(columns='Usual side of head for mobile phone use'),
    dummies
], axis=1)

print('NA\n', (device_use_i2_encode_dummy.isna()).sum().sort_values(ascending=False))
print('MIN\n', device_use_i2_encode_dummy.min())
print('NEG\n', (device_use_i2_encode_dummy < 0).sum().sort_values(ascending=False))

dummies.columns.tolist()

For other columns, as the features are ordinal, ie reflect magnitude, they will not be dummy-encoded. Therefore, non-responders will be encoded as NA.

In [ ]:
# Assign NAs to non-responders
device_use_i2_encode_dummy.replace(-1, np.nan, inplace=True)
device_use_i2_encode_dummy.replace(-3, np.nan, inplace=True)

print('NA\n', (device_use_i2_encode_dummy.isna()).sum().sort_values(ascending=False))
print('MIN\n', device_use_i2_encode_dummy.min())
print('NEG\n', (device_use_i2_encode_dummy < 0).sum().sort_values(ascending=False))

device_use_i2_encode_dummy.to_csv(os.path.join(data_path, 'electronic_device_use.csv'), index=False)

In [ ]:
# Match features to the target and compute sample sizes
folds = range(0,5)
modalities = ['electronic_device_use']
for modality in modalities:
    for fold in folds:
        base_path = f'/UK_BB/brainbody/lifestyle'
        folds_path = os.path.join(base_path, f'folds/fold_{fold}')
        suppl_path = os.path.join(folds_path, 'suppl')
        scaling_path = os.path.join(folds_path, 'scaling')
        models_path = os.path.join(folds_path, 'models')
        g_pred_path = os.path.join(folds_path, 'g_pred')
        
        os.makedirs(folds_path, exist_ok=True)
        os.makedirs(suppl_path, exist_ok=True)
        os.makedirs(scaling_path, exist_ok=True)
        os.makedirs(models_path, exist_ok=True)
        os.makedirs(g_pred_path, exist_ok=True)

        g_train = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_train_with_id_{fold}.csv')
        g_test = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_test_with_id_{fold}.csv')
        features = pd.read_csv(os.path.join(base_path, f'data/{modality}.csv'))
        print('Features shape', features.shape)

        feature_columns = features.drop(columns='eid').columns

        # Merge features with targets
        train_merge_all = pd.merge(features, g_train, on='eid').reset_index(drop=True)
        test_merge_all = pd.merge(features, g_test, on='eid').reset_index(drop=True)

        # Save merged data
        train_merge_all.to_csv(os.path.join(suppl_path, f'{modality}_train_feat_targ_fold_{fold}.csv'), index=False)
        test_merge_all.to_csv(os.path.join(suppl_path, f'{modality}_test_feat_targ_fold_{fold}.csv'), index=False)

        print(f'==== Train shape ====\n {modality} - Fold {fold}', train_merge_all.shape)
        print(f'==== Test shape ====\n {modality} - Fold {fold}', test_merge_all.shape)

        # Extract features and save
        train_merge_all[feature_columns].to_csv(os.path.join(suppl_path, f'{modality}_train_fold_{fold}.csv'), index=False)
        test_merge_all[feature_columns].to_csv(os.path.join(suppl_path, f'{modality}_test_fold_{fold}.csv'), index=False)

        # Scale features
        print('Scaling')
        scaler_features = StandardScaler()
        features_train_scaled = scaler_features.fit_transform(train_merge_all[feature_columns])
        features_test_scaled = scaler_features.transform(test_merge_all[feature_columns])
        pd.DataFrame(features_train_scaled, columns=feature_columns).to_csv(os.path.join(scaling_path, f'{modality}_train_scaled_fold_{fold}.csv'), index=False)
        pd.DataFrame(features_test_scaled, columns=feature_columns).to_csv(os.path.join(scaling_path, f'{modality}_test_scaled_fold_{fold}.csv'), index=False)

        # Save scaler
        with open(os.path.join(scaling_path, f'{modality}_scaler_features_fold_{fold}.pkl'), "wb") as f:
            pickle.dump(scaler_features, f)

In [ ]:
# Check if there are columns with constant values (e.g., all 0)
def is_constant(column):
    """Check if all values in a column are the same."""
    return np.all(column == column.iloc[0])

def find_constant_columns(df, df_name):
    constant_columns = []
    
    print(f"\nChecking for constant columns in {df_name}...")
    
    for column in df.columns:
        if is_constant(df[column]):
            constant_columns.append(column)
            print(f"✅ Column '{column}' is constant.")
        else:
            print(f"❌ Column '{column}' is NOT constant.")
    
    if constant_columns:
        print(f"\n🔍 Found {len(constant_columns)} constant column(s) in {df_name}: {constant_columns}")
    else:
        print(f"\n🎉 No constant columns found in {df_name}!")
    
    return constant_columns

constant_columns = find_constant_columns(device_use_i2_encode_dummy, "electronic_device_use")

# Alcohol

In [ ]:
df_alcohol = ukbiobank.utils.utils.loadCsv(ukbio=ukb, fields=['eid',
20117, #Alcohol drinker status
1558, #Alcohol intake frequency.
3731, #Former alcohol drinker
4407, #Average monthly red wine intake
4418, #Average monthly champagne plus white wine intake
4429, #Average monthly beer plus cider intake
4440, #Average monthly spirits intake
4451, #Average monthly fortified wine intake
4462, #Average monthly intake of other alcoholic drinks
1568, #Average weekly red wine intake
1578, #Average weekly champagne plus white wine intake
1588, #Average weekly beer plus cider intake
1598, #Average weekly spirits intake
1608, #Average weekly fortified wine intake
5364, #Average weekly intake of other alcoholic drinks
1618, #Alcohol usually taken with meals
1628, #Alcohol intake versus 10 years previously
2664, #Reason for reducing amount of alcohol drunk
3859, #Reason former drinker stopped drinking alcohol
])
df_alcohol_names = ukbiobank.utils.utils.fieldIdsToNames(ukbio=ukb, df=df_alcohol)
df_alcohol_names.to_csv(os.path.join(data_path, 'alcohol_vars.csv'), index=False)

In [ ]:
print('% missing')
with pd.option_context('display.max_rows', None):
    display(((df_alcohol_names.isna().sum() / len(df_alcohol_names)).round(2)*100).sort_values(ascending=False))

In [ ]:
# Filter Instance 2
df_alcohol_names = pd.read_csv(os.path.join(data_path, 'alcohol_vars.csv'))
alcohol_i2 = df_alcohol_names[['eid'] + df_alcohol_names.filter(regex=r'2\.\d$').columns.tolist()]
alcohol_i2.columns = alcohol_i2.columns.str.replace('-2.0', '')
alcohol_i2.columns = alcohol_i2.columns.str.replace('Alcohol intake frequency.', 'Alcohol intake frequency')
alcohol_i2 = alcohol_i2.dropna(subset=['Alcohol drinker status', 'Alcohol intake frequency']).reset_index(drop=True)
alcohol_i2.to_csv(os.path.join(data_path, 'alcohol_i2_raw.csv'), index=False)
print('SHAPE:', alcohol_i2.shape)
with pd.option_context('display.max_rows', None):
    display(((alcohol_i2.isna().sum() / len(alcohol_i2)).round(2)*100).sort_values(ascending=False))

In [ ]:
alcohol_encode = alcohol_i2.copy()

1. Questionnaire will not proceed if 'Alcohol drinker status' is never

In [ ]:
# Alcohol drinker status
alcohol_encode.loc[
    alcohol_encode['Alcohol drinker status'] == 0, alcohol_encode.drop(columns=['eid', 'Alcohol drinker status']).columns] = alcohol_encode.loc[
    alcohol_encode['Alcohol drinker status'] == 0, alcohol_encode.drop(columns=['eid', 'Alcohol drinker status']).columns].fillna(0)

2. 'Former alcohol drinker' was collected from participants who indicated they never drink alcohol, as defined by their answers to Field 'Alcohol intake frequency'

In [ ]:
# Former alcohol drinker
alcohol_encode.loc[
    alcohol_encode['Alcohol intake frequency'] != 6, 'Former alcohol drinker'] = alcohol_encode.loc[
        alcohol_encode['Alcohol intake frequency'] != 6, 'Former alcohol drinker'].fillna(0)
#6 = Never

In [ ]:
#1        Daily or almost daily
#2        Three or four times a week
#3        Once or twice a week
#4        One to three times a month
#5        Special occasions only
#6        Never
#-3       Prefer not to answer

3. Collected from participants who indicated they drink alcohol on *special occasions* or *one to three times a month*

In [ ]:
# Average monthly alcohol intake
monthly_intake = [
    'Average monthly red wine intake', 
    'Average monthly champagne plus white wine intake',
    'Average monthly beer plus cider intake',
    'Average monthly spirits intake',
    'Average monthly fortified wine intake',
    'Average monthly intake of other alcoholic drinks']

# Fill NA with 0 for rows where Alcohol intake frequency is neither 4 nor 5
alcohol_encode.loc[
    (alcohol_encode['Alcohol intake frequency'] != 4) & (alcohol_encode['Alcohol intake frequency'] != 5), monthly_intake
] = alcohol_encode.loc[
    (alcohol_encode['Alcohol intake frequency'] != 4) & (alcohol_encode['Alcohol intake frequency'] != 5), monthly_intake
].fillna(0)

#5 =  Special occasions only
#4 = One to three times a month

4. Collected from participants who indicated they drink alcohol more often than *once or twice a week*

In [ ]:
# Average weekly alcohol intake
weekly_intake = ['Average weekly red wine intake', 
                'Average weekly champagne plus white wine intake',
                'Average weekly spirits intake',
                'Average weekly fortified wine intake',
                'Average weekly intake of other alcoholic drinks']

alcohol_encode.loc[alcohol_encode['Alcohol intake frequency'].isin([3,4,5,6,-3]), weekly_intake] = alcohol_encode.loc[
        alcohol_encode['Alcohol intake frequency'].isin([3,4,5,6,-3]), weekly_intake].fillna(0)

#1 Daily or almost daily - were asked
#2 =  Three or four times a week - were asked
#3 = Once or twice a week - more often than that  - were NOT asked
#4 = One to three times a month - were NOT asked
#5 = Special occasions only  - were NOT asked
#6 = Never  - were NOT asked
#-3 = Prefer not to answer  - were NOT asked

5. Collected from participants who indicated they drink alcohol more often than *one to three times per month*

In [ ]:
# Average weekly beer plus cider intake
alcohol_encode.loc[
    alcohol_encode['Alcohol intake frequency'].isin([4,5,6,-3]), 'Average weekly beer plus cider intake'] = alcohol_encode.loc[
        alcohol_encode['Alcohol intake frequency'].isin([4,5,6,-3]), 'Average weekly beer plus cider intake'].fillna(0)

#1 Daily or almost daily - were asked
#2 =  Three or four times a week - were asked
#3 = Once or twice a week  - were asked
#4 = One to three times a month  - more often than that  - were NOT asked
#5 = Special occasions only  - were NOT asked
#6 = Never  - were NOT asked
#-3 = Prefer not to answer  - were NOT asked

6. Collected from participants who indicated they drink alcohol

In [ ]:
# Alcohol usually taken with meals
meals_intake = ['Alcohol usually taken with meals', 
                'Alcohol intake versus 10 years previously']
alcohol_encode.loc[
    alcohol_encode['Alcohol intake frequency'].isin([6, -3]), meals_intake] = alcohol_encode.loc[
        alcohol_encode['Alcohol intake frequency'].isin([6, -3]), meals_intake].fillna(0)

#1 = Daily or almost daily
#2 = Three or four times a week
#3 = Once or twice a week
#4 = One to three times a month
#5 = Special occasions only
#6 = Never - they were NOT asked
#-3 = Prefer not to answer - they were NOT asked

7. Collected from participants who indicated they drink alcohol and they drink less nowadays than 10 years ago (in 'Alcohol intake versus 10 years previously')

In [ ]:
# Reason for reducing amount of alcohol drunk
alcohol_encode.loc[
    alcohol_encode['Alcohol intake frequency'].isin([6, -3]), 'Reason for reducing amount of alcohol drunk'] = alcohol_encode.loc[
    alcohol_encode['Alcohol intake frequency'].isin([6, -3]), 'Reason for reducing amount of alcohol drunk'].fillna(0)

alcohol_encode.loc[
    alcohol_encode['Alcohol intake versus 10 years previously']!= 3, 'Reason for reducing amount of alcohol drunk'] = alcohol_encode.loc[
    alcohol_encode['Alcohol intake versus 10 years previously']!= 3, 'Reason for reducing amount of alcohol drunk'].fillna(0)


# Fill NaN with 0 for participants with Alcohol intake frequency as 6 or -3 and Alcohol intake versus 10 years previously not 3


# Alcohol intake frequency
#1 = Daily or almost daily
#2 = Three or four times a week
#3 = Once or twice a week
#4 = One to three times a month
#5 = Special occasions only
#6 = Never - they were NOT asked
#-3 = Prefer not to answer - they were NOT asked

#####
# Alcohol intake versus 10 years previously
#1 = More nowadays
#2 = About the same
#3 = Less nowadays  - WERE asked
#-1 = Do not know
#-3 = Prefer not to answer

8. Collected from participants who indicated that they currently never drink alcohol and they previously drank alcohol (in 'Former alcohol drinker')

In [ ]:
# Reason former drinker stopped drinking alcohol
alcohol_encode.loc[
    (alcohol_encode['Alcohol drinker status'] != 0) &  (alcohol_encode['Former alcohol drinker'] != 1), 'Reason former drinker stopped drinking alcohol'] = alcohol_encode.loc[
        (alcohol_encode['Alcohol drinker status'] != 0) &  (alcohol_encode['Former alcohol drinker'] != 1), 'Reason former drinker stopped drinking alcohol'].fillna(0)

#1 = Daily or almost daily
#2 = Three or four times a week
#3 = Once or twice a week
#4 = One to three times a month
#5 = Special occasions only
#6 = Never - only these were asked
#-3 = Prefer not to answer

#####

# Former alcohol drinker
#1 = Yes - only these were asked
#0 = No
#-3 = Prefer not to answer

###
#-3	Prefer not to answer
#0	Never
#1	Previous
#2	Current

In [ ]:
with pd.option_context('display.max_rows', None):
    display(alcohol_encode.isna().sum().sort_values(ascending=False))

##### Dummy-encoding

In [ ]:
alcohol_dummy = alcohol_encode.copy()

In [ ]:
# Alcohol drinker status

alco_status_mapping = {
-3:	'Prefer not to answer',
0:	'Never',
1:	'Previous',
2:	'Current'}


alcohol_dummy['Alcohol drinker status'] = (
    alcohol_dummy['Alcohol drinker status']
    .apply(pd.to_numeric, errors='coerce')
    .astype('Int64')
)

# Validate values (fail on unexpected)
valid_values = {-3, 0, 1, 2}
invalid_mask = ~alcohol_dummy['Alcohol drinker status'].isin(valid_values)
if invalid_mask.any():
    raise ValueError(f"Unexpected values: {alcohol_dummy.loc[invalid_mask, 'Alcohol drinker status'].unique()}")

# Map to labels
alcohol_dummy['Alcohol drinker status'] = alcohol_dummy['Alcohol drinker status'].map(alco_status_mapping)

# One-hot encode
dummies = pd.get_dummies(
    alcohol_dummy['Alcohol drinker status'],
    prefix='Alcohol drinker status',
    prefix_sep=' - '  # Safer than manual replace
)

# Ensure all expected categories exist
expected_categories = ['Prefer not to answer', 'Never', 'Previous', 'Current']
for cat in expected_categories:
    col_name = f"Alcohol drinker status - {cat}"
    if col_name not in dummies.columns:
        dummies[col_name] = 0

# Drop original & concat
alcohol_dummy = alcohol_dummy.drop(columns='Alcohol drinker status')
alcohol_dummy = pd.concat([alcohol_dummy, dummies], axis=1)

print('SHAPE:', alcohol_dummy.shape)
print('COLUMNS:', alcohol_dummy.columns.tolist())

In [ ]:
# Alcohol intake frequency
alcohol_dummy.loc[:, ['Alcohol intake frequency']] = alcohol_dummy.loc[:, ['Alcohol intake frequency']].replace({
1: 5, #Daily or almost daily
2: 4, #Three or four times a week
3: 3, #Once or twice a week
4: 2, #One to three times a month
5: 1, #Special occasions only
6: 0 #Never
})

# was:
# 1 Daily or almost daily
# 2 Three or four times a week
# 3 Once or twice a week
# 4 One to three times a month
# 5 Special occasions only
# 6 Never
# -3 Prefer not to answer

In [ ]:
# Alcohol usually taken with meals
alcohol_dummy.loc[:, ['Alcohol usually taken with meals']] = alcohol_dummy.loc[:, ['Alcohol usually taken with meals']].replace({
-6: 0.5  # 'It varies', 1 = Yes, 0 = No
})

In [ ]:
# Alcohol intake versus 10 years previously
ten_years_mapping = {
0:	"Don't drink alcohol",    
1:	'More nowadays',
2:	'About the same',
3:	'Less nowadays',
-1:	'Do not know / Prefer not to answer',
-3:	'Do not know / Prefer not to answer'
}

# Convert to numeric (strict)
alcohol_dummy['Alcohol intake versus 10 years previously'] = (
    alcohol_dummy['Alcohol intake versus 10 years previously']
    .apply(pd.to_numeric, errors='coerce')
    .astype('Int64')
)

# Validate values (fail on unexpected)
valid_values = {0, 1, 2, 3, -1, -3}
invalid_mask = ~alcohol_dummy['Alcohol intake versus 10 years previously'].isin(valid_values)
if invalid_mask.any():
    raise ValueError(f"Unexpected values: {alcohol_dummy.loc[invalid_mask, 'Alcohol intake versus 10 years previously'].unique()}")

# Map to labels
alcohol_dummy['Alcohol intake versus 10 years previously'] = (
    alcohol_dummy['Alcohol intake versus 10 years previously']
    .map(ten_years_mapping)
)

# One-hot encode
dummies = pd.get_dummies(
    alcohol_dummy['Alcohol intake versus 10 years previously'],
    prefix='Alcohol intake versus 10 years previously',
    prefix_sep=' - '
)

# Ensure all expected categories exist
expected_categories = [
    "Don't drink alcohol", 
    'More nowadays',
    'About the same',
    'Less nowadays',
    'Do not know / Prefer not to answer'
]
for cat in expected_categories:
    col_name = f"Alcohol intake versus 10 years previously - {cat}"
    if col_name not in dummies.columns:
        dummies[col_name] = 0

# Drop original & concat
alcohol_dummy = alcohol_dummy.drop(columns='Alcohol intake versus 10 years previously')
alcohol_dummy = pd.concat([alcohol_dummy, dummies], axis=1)

print('SHAPE:', alcohol_dummy.shape)
print('COLUMNS:', alcohol_dummy.columns.tolist())

#1	More nowadays
#2	About the same
#3	Less nowadays
#-1	Do not know
#-3	Prefer not to answer"

In [ ]:
# Reason for reducing amount of alcohol drunk, Reason former drinker stopped drinking alcohol
alco_reduce = ['Reason for reducing amount of alcohol drunk', 'Reason former drinker stopped drinking alcohol']

# Convert to numeric + Int64
alcohol_dummy[alco_reduce] = (
    alcohol_dummy[alco_reduce]
    .apply(pd.to_numeric, errors='coerce')
    .astype('Int64')
)

# Define the mapping for alcohol reduction reasons
alco_reduce_mapping = {
    0:	"Don't drink alcohol",   
    1: 'Illness or ill health',
    2: "Doctor's advice",
    3: 'Health precaution',
    4: 'Financial reasons',
    5: 'Other reason',
    -1: 'Do not know / Prefer not to answer',
    -3: 'Do not know / Prefer not to answer'
}

# Validate values (fail on unexpected)
valid_values = {0, 1, 2, 3, 4, 5, -1, -3}
for col in alco_reduce:
    invalid_mask = ~alcohol_dummy[col].isin(valid_values)
    if invalid_mask.any():
        raise ValueError(f"Unexpected values in '{col}': {alcohol_dummy.loc[invalid_mask, col].unique()}")

# Map to labels
for col in alco_reduce:
    alcohol_dummy[col] = alcohol_dummy[col].map(alco_reduce_mapping)

# Dummy encode each column (with forced categories)
dummies_list = []
expected_categories = [
    "Don't drink alcohol",   
    'Illness or ill health',
    "Doctor's advice",
    'Health precaution',
    'Financial reasons',
    'Other reason',
    'Do not know / Prefer not to answer'
]

for col in alco_reduce:
    dummies = pd.get_dummies(
        alcohol_dummy[col],
        prefix=col,
        prefix_sep=' - '
    )
    # Ensure all expected categories exist
    for cat in expected_categories:
        col_name = f"{col} - {cat}"
        if col_name not in dummies.columns:
            dummies[col_name] = 0
    dummies_list.append(dummies)

# Combine and drop original columns
dummies = pd.concat(dummies_list, axis=1)
alcohol_dummy = pd.concat([alcohol_dummy, dummies], axis=1)
alcohol_dummy.drop(columns=alco_reduce, inplace=True)

print('Shape:', alcohol_dummy.shape)
print('Columns:', alcohol_dummy.columns.tolist())

In [ ]:
# Count NA and negative values
print('Shape', alcohol_dummy.shape)
print('NA\n', (alcohol_dummy.isna()).sum().sort_values(ascending=False))
print('Number of -1 responses\n', (alcohol_dummy == -1).sum().sort_values(ascending=False))
print('Number of -3 responses\n', (alcohol_dummy == -3).sum().sort_values(ascending=False))
print('Number of -6 responses\n', (alcohol_dummy == -6).sum().sort_values(ascending=False))

In [ ]:
# Assign NAs to non-responders
alcohol_dummy.replace(-1, np.nan, inplace=True)
alcohol_dummy.replace(-3, np.nan, inplace=True)

print('NA\n', (alcohol_dummy.isna()).sum().sort_values(ascending=False))
print('MIN\n', alcohol_dummy.min())
print('NEG\n', (alcohol_dummy < 0).sum().sort_values(ascending=False))

alcohol_dummy.to_csv(os.path.join(data_path, 'alcohol.csv'), index=False)

In [ ]:
# Match features to the target and compute sample sizes
folds = range(0,5)
modalities = ['alcohol']
for modality in modalities:
    for fold in folds:
        base_path = f'/UK_BB/brainbody/lifestyle'
        folds_path = os.path.join(base_path, f'folds/fold_{fold}')
        suppl_path = os.path.join(folds_path, 'suppl')
        scaling_path = os.path.join(folds_path, 'scaling')
        models_path = os.path.join(folds_path, 'models')
        g_pred_path = os.path.join(folds_path, 'g_pred')
        
        os.makedirs(folds_path, exist_ok=True)
        os.makedirs(suppl_path, exist_ok=True)
        os.makedirs(scaling_path, exist_ok=True)
        os.makedirs(models_path, exist_ok=True)
        os.makedirs(g_pred_path, exist_ok=True)

        g_train = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_train_with_id_{fold}.csv')
        g_test = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_test_with_id_{fold}.csv')
        features = pd.read_csv(os.path.join(base_path, f'data/{modality}.csv'))
        print('Features shape', features.shape)

        feature_columns = features.drop(columns='eid').columns

        # Merge features with targets
        train_merge_all = pd.merge(features, g_train, on='eid').reset_index(drop=True)
        test_merge_all = pd.merge(features, g_test, on='eid').reset_index(drop=True)

        # Save merged data
        train_merge_all.to_csv(os.path.join(suppl_path, f'{modality}_train_feat_targ_fold_{fold}.csv'), index=False)
        test_merge_all.to_csv(os.path.join(suppl_path, f'{modality}_test_feat_targ_fold_{fold}.csv'), index=False)

        print(f'==== Train shape ====\n {modality} - Fold {fold}', train_merge_all.shape)
        print(f'==== Test shape ====\n {modality} - Fold {fold}', test_merge_all.shape)

        # Extract features and save
        train_merge_all[feature_columns].to_csv(os.path.join(suppl_path, f'{modality}_train_fold_{fold}.csv'), index=False)
        test_merge_all[feature_columns].to_csv(os.path.join(suppl_path, f'{modality}_test_fold_{fold}.csv'), index=False)

        # Scale features
        print('Scaling')
        scaler_features = StandardScaler()
        features_train_scaled = scaler_features.fit_transform(train_merge_all[feature_columns])
        features_test_scaled = scaler_features.transform(test_merge_all[feature_columns])
        pd.DataFrame(features_train_scaled, columns=feature_columns).to_csv(os.path.join(scaling_path, f'{modality}_train_scaled_fold_{fold}.csv'), index=False)
        pd.DataFrame(features_test_scaled, columns=feature_columns).to_csv(os.path.join(scaling_path, f'{modality}_test_scaled_fold_{fold}.csv'), index=False)

        # Save scaler
        with open(os.path.join(scaling_path, f'{modality}_scaler_features_fold_{fold}.pkl'), "wb") as f:
            pickle.dump(scaler_features, f)

In [ ]:
# Check if there are columns with constant values (e.g., all 0)
constant_columns = find_constant_columns(alcohol_dummy, "aclohol")

# Sun exposure

In [ ]:
df_sun = ukbiobank.utils.utils.loadCsv(ukbio=ukb, fields=['eid',
1050, #Time spend outdoors in summer
1060, #Time spent outdoors in winter
1717, #Skin colour
1727, #Ease of skin tanning
1737, #Childhood sunburn occasions
1747, #Hair colour (natural, before greying)
1757, #Facial ageing
2267, #Use of sun/uv protection
2277 #Frequency of solarium/sunlamp use
])
df_sun_names = ukbiobank.utils.utils.fieldIdsToNames(ukbio=ukb, df=df_sun)
df_sun_names.to_csv(os.path.join(data_path, 'sun_exposure_vars.csv'), index=False)

In [ ]:
print('% missing')
with pd.option_context('display.max_rows', None):
    display(((df_sun_names.isna().sum() / len(df_sun_names)).round(2)*100).sort_values(ascending=False))

In [ ]:
# Filter Instance 2
df_sun_names = pd.read_csv(os.path.join(data_path, 'sun_exposure_vars.csv'))
sun_i2 = df_sun_names[['eid'] + df_sun_names.filter(regex=r'2\.\d$').columns.tolist()]
sun_i2.columns = sun_i2.columns.str.replace('-2.0', '')
print('SHAPE:', sun_i2.shape)
with pd.option_context('display.max_rows', None):
    display(((sun_i2.isna().sum() / len(sun_i2)).round(2)*100).sort_values(ascending=False))

In Instance 2, the following questions were *'dropped from the touchscreen protocol on 24/10/2016'*:

- Skin colour
- Ease of skin tanning
- Childhood sunburn occasions
- Hair colour (natural, before greying)

In [ ]:
sun_i2 = sun_i2.drop(columns=['Skin colour',
                              'Ease of skin tanning',
                              'Childhood sunburn occasions',
                              'Hair colour (natural, before greying)']).dropna().reset_index(drop=True)
print('SHAPE:', sun_i2.shape)
print('NAs:', ((sun_i2.isna().sum() / len(sun_i2)).round(2)*100).sort_values(ascending=False))
sun_i2.to_csv(os.path.join(data_path, 'sun_i2_raw.csv'), index=False)

In [ ]:
# Filter Instance 0
sun_i0 = df_sun_names[['eid'] + df_sun_names.filter(regex=r'0\.\d$').columns.tolist()]
sun_i0.columns = sun_i0.columns.str.replace('-0.0', '')
print('SHAPE:', sun_i0.shape)
with pd.option_context('display.max_rows', None):
    display(((sun_i0.isna().sum() / len(sun_i0)).round(2)*100).sort_values(ascending=False))

with pd.option_context('display.max_rows', None):
    display(sun_i0.isna().sum().sort_values(ascending=False))

In [ ]:
sun_i0 = sun_i0.dropna().reset_index(drop=True)
print('SHAPE:', sun_i0.shape)
print('NAs:', ((sun_i0.isna().sum() / len(sun_i0)).round(2)*100).sort_values(ascending=False))
sun_i0.to_csv(os.path.join(data_path, 'sun_i0_raw.csv'), index=False)

##### Handle the data

In [ ]:
sun_i0_encode = sun_i0.copy()
sun_i2_encode = sun_i2.copy()

In [ ]:
# Replace -10  with 0.5
sun_i0_encode.loc[:, [
    'Time spend outdoors in summer',
    'Time spent outdoors in winter',
    'Frequency of solarium/sunlamp use']] = sun_i0_encode.loc[:, [
    'Time spend outdoors in summer',
    'Time spent outdoors in winter',
    'Frequency of solarium/sunlamp use']].replace({-10: 0.5, #Less than 1
})
print('Number of -10 responses in Instance 0\n', (sun_i0_encode == -10).sum().sort_values(ascending=False))


sun_i2_encode.loc[:, [
    'Time spend outdoors in summer',
    'Time spent outdoors in winter',
    'Frequency of solarium/sunlamp use']] = sun_i2_encode.loc[:, [
    'Time spend outdoors in summer',
    'Time spent outdoors in winter',
    'Frequency of solarium/sunlamp use']].replace({-10: 0.5, #Less than 1
})
print('Number of -10 responses in Instance 2\n', (sun_i2_encode == -10).sum().sort_values(ascending=False))

In [ ]:
# Use of sun/uv protection
sun_i0_encode.loc[:, ['Use of sun/uv protection']] = sun_i0_encode.loc[:, [
    'Use of sun/uv protection']].replace({5: 0, #Do not go out in sunshine
})

sun_i2_encode.loc[:, ['Use of sun/uv protection']] = sun_i2_encode.loc[:, [
    'Use of sun/uv protection']].replace({5: 0, #Do not go out in sunshine
})

#1	Never/rarely
#2	Sometimes
#3	Most of the time
#4	Always
#5	Do not go out in sunshine
#-1	Do not know
#-3	Prefer not to answer"

In [ ]:
# Dummy encode: Instance 0
###################################################################################
# Skin colour
skin_colour_mapping = {
    1: 'Very fair',
    2: 'Fair',
    3: 'Light olive',
    4: 'Dark olive',
    5: 'Brown',
    6: 'Black',
    -1: 'Do not know / Prefer not to answer',
    -3: 'Do not know / Prefer not to answer'
}

# Validate before mapping
valid_values = set(skin_colour_mapping.keys())
invalid_mask = ~sun_i0_encode['Skin colour'].isin(valid_values) & sun_i0_encode['Skin colour'].notna()
if invalid_mask.any():
    raise ValueError(f"Invalid skin colour codes: {sun_i0_encode.loc[invalid_mask, 'Skin colour'].unique()}")

sun_i0_encode['Skin colour'] = sun_i0_encode['Skin colour'].map(skin_colour_mapping)
dummies = pd.get_dummies(
    sun_i0_encode['Skin colour'], 
    prefix='Skin colour',
    prefix_sep=' - '
)
sun_i0_encode = sun_i0_encode.drop(columns='Skin colour')
sun_i0_encode = pd.concat([sun_i0_encode, dummies], axis=1)
print('SHAPE:', sun_i0_encode.shape)

###################################################################################
# Hair colour (natural, before greying)
hair_colour_mapping = {
    1: 'Blonde',
    2: 'Red',
    3: 'Light brown',
    4: 'Dark brown',
    5: 'Black',
    6: 'Other',
    -1: 'Do not know / Prefer not to answer',
    -3: 'Do not know / Prefer not to answer'
}

# Validate before mapping
valid_values = set(hair_colour_mapping.keys())
invalid_mask = ~sun_i0_encode['Hair colour (natural, before greying)'].isin(valid_values) & sun_i0_encode['Hair colour (natural, before greying)'].notna()
if invalid_mask.any():
    raise ValueError(f"Invalid hair colour codes: {sun_i0_encode.loc[invalid_mask, 'Hair colour (natural, before greying)'].unique()}")

sun_i0_encode['Hair colour (natural, before greying)'] = sun_i0_encode['Hair colour (natural, before greying)'].map(hair_colour_mapping)
dummies = pd.get_dummies(
    sun_i0_encode['Hair colour (natural, before greying)'],
    prefix='Hair colour (natural, before greying)',
    prefix_sep=' - '
)
sun_i0_encode = sun_i0_encode.drop(columns='Hair colour (natural, before greying)')
sun_i0_encode = pd.concat([sun_i0_encode, dummies], axis=1)
print('SHAPE:', sun_i0_encode.shape)

###################################################################################
# Facial ageing
face_mapping = {
    1: 'Younger than you are',
    2: 'Older than you are',
    3: 'About your age',
    -1: 'Do not know / Prefer not to answer',
    -3: 'Do not know / Prefer not to answer'
}

# Validate before mapping
valid_values = set(face_mapping.keys())
invalid_mask = ~sun_i0_encode['Facial ageing'].isin(valid_values) & sun_i0_encode['Facial ageing'].notna()
if invalid_mask.any():
    raise ValueError(f"Invalid facial ageing codes: {sun_i0_encode.loc[invalid_mask, 'Facial ageing'].unique()}")

sun_i0_encode['Facial ageing'] = sun_i0_encode['Facial ageing'].map(face_mapping)
dummies = pd.get_dummies(
    sun_i0_encode['Facial ageing'],
    prefix='Facial ageing',
    prefix_sep=' - '
)
sun_i0_encode = sun_i0_encode.drop(columns='Facial ageing')
sun_i0_encode = pd.concat([sun_i0_encode, dummies], axis=1)
print('SHAPE:', sun_i0_encode.shape)

In [ ]:
# Dummy encode: Instance 2
###################################################################################
# Skin colour - dropped from Instance 2 
###################################################################################
# Hair colour (natural, before greying) - dropped from Instance 2 
###################################################################################
# Facial ageing
face_mapping = {
    1: 'Younger than you are',
    2: 'Older than you are',
    3: 'About your age',
    -1: 'Do not know / Prefer not to answer',
    -3: 'Do not know / Prefer not to answer'
}

# Validate before mapping
valid_values = set(face_mapping.keys())
invalid_mask = ~sun_i2_encode['Facial ageing'].isin(valid_values) & sun_i2_encode['Facial ageing'].notna()
if invalid_mask.any():
    raise ValueError(f"Invalid facial ageing codes: {sun_i2_encode.loc[invalid_mask, 'Facial ageing'].unique()}")

sun_i2_encode['Facial ageing'] = sun_i2_encode['Facial ageing'].map(face_mapping)
dummies = pd.get_dummies(sun_i2_encode['Facial ageing'], prefix='Facial ageing').astype(int)
dummies.columns = [col.replace("_", " - ") for col in dummies.columns]
sun_i2_encode = sun_i2_encode.drop(columns='Facial ageing')
sun_i2_encode = pd.concat([sun_i2_encode, dummies], axis=1)
print('SHAPE:', sun_i2_encode.shape)
print('Columns:', [col for col in sun_i2_encode.columns if col.startswith('Facial ageing')])

#1	Younger than you are
#2	Older than you are
#3	About your age
#-1	Do not know
#-3	Prefer not to answer"

In [ ]:
# Count negative values
print('MIN, Instance 0\n', sun_i0_encode.min())
print('MAX, Instance 0\n', sun_i0_encode.max())

print('Number of -1 responses, Instance 0\n', (sun_i0_encode == -1).sum().sort_values(ascending=False))
print('Number of -3 responses, Instance 0\n', (sun_i0_encode == -3).sum().sort_values(ascending=False))
print('Number of -10 responses, Instance 0\n', (sun_i0_encode == -10).sum().sort_values(ascending=False))
'''
print('MIN, Instance 2\n', sun_i2_encode.min())
print('MAX, Instance 2\n', sun_i2_encode.max())

print('Number of -1 responses, Instance 2\n', (sun_i2_encode == -1).sum().sort_values(ascending=False))
print('Number of -3 responses, Instance 2\n', (sun_i2_encode == -3).sum().sort_values(ascending=False))
print('Number of -10 responses, Instance 2\n', (sun_i2_encode == -10).sum().sort_values(ascending=False))
'''

In [ ]:
# Count NA and negative values
print('Shape, Instance 0', sun_i0_encode.shape)
print('NA, Instance 0\n', (sun_i0_encode.isna()).sum().sort_values(ascending=False))

In [ ]:
# Replace -1 and -3 (non-respondes) with NA in orinal/magnitude variables
non_responders_i0 = ['Time spend outdoors in summer',
                     'Time spent outdoors in winter',
                     'Ease of skin tanning',
                     'Childhood sunburn occasions',
                     'Use of sun/uv protection',
                     'Frequency of solarium/sunlamp use']

non_responders_i2 = ['Time spend outdoors in summer',
                     'Time spent outdoors in winter',
                     #'Ease of skin tanning', dropped
                     #'Childhood sunburn occasions',
                     'Use of sun/uv protection',
                     'Frequency of solarium/sunlamp use']

sun_i0_encode[non_responders_i0] = sun_i0_encode[non_responders_i0].replace([-1, -3], np.nan)
sun_i2_encode[non_responders_i2] = sun_i2_encode[non_responders_i2].replace([-1, -3], np.nan)

In [ ]:
# Count NA and negative values
print('Shape, Instance 0', sun_i0_encode.shape)
print('NA, Instance 0\n', (sun_i0_encode.isna()).sum().sort_values(ascending=False))
print('MIN, Instance 0\n', sun_i0_encode.min())
print('MAX, Instance 0\n', sun_i0_encode.max())

print('Number of -1 responses, Instance 0\n', (sun_i0_encode == -1).sum().sort_values(ascending=False))
print('Number of -3 responses, Instance 0\n', (sun_i0_encode == -3).sum().sort_values(ascending=False))
print('Number of -10 responses, Instance 0\n', (sun_i0_encode == -10).sum().sort_values(ascending=False))

print('Shape, Instance 2', sun_i2_encode.shape)
print('NA, Instance 2\n', (sun_i2_encode.isna()).sum().sort_values(ascending=False))
print('MIN, Instance 2\n', sun_i2_encode.min())
print('MAX, Instance 2\n', sun_i2_encode.max())

print('Number of -1 responses, Instance 2\n', (sun_i2_encode == -1).sum().sort_values(ascending=False))
print('Number of -3 responses, Instance 2\n', (sun_i2_encode == -3).sum().sort_values(ascending=False))
print('Number of -10 responses, Instance 2\n', (sun_i2_encode == -10).sum().sort_values(ascending=False))

In [ ]:
sun_i0_encode.to_csv(os.path.join(data_path, 'sun_i0.csv'), index=False)
sun_i2_encode.to_csv(os.path.join(data_path, 'sun_i2.csv'), index=False)

In [ ]:
# Check if there are columns with constant values (e.g., all 0)
def is_constant(column):
    """Check if all values in a column are the same."""
    return np.all(column == column.iloc[0])

def find_constant_columns(df, df_name):
    constant_columns = []
    
    print(f"\nChecking for constant columns in {df_name}...")
    
    for column in df.columns:
        if is_constant(df[column]):
            constant_columns.append(column)
            print(f"✅ Column '{column}' is constant.")
        else:
            print(f"❌ Column '{column}' is NOT constant.")
    
    if constant_columns:
        print(f"\n🔍 Found {len(constant_columns)} constant column(s) in {df_name}: {constant_columns}")
    else:
        print(f"\n🎉 No constant columns found in {df_name}!")
    
    return constant_columns

constant_columns_i0 = find_constant_columns(sun_i0_encode, "sun_i0_encode")
constant_columns_i2 = find_constant_columns(sun_i2_encode, "sun_i2_encode")

In [ ]:
# Match features to the target and compute sample sizes
folds = range(0,5)
modalities = ['sun_i0',
              'sun_i2']

modality_observations = {modality: {'train': 0, 'test': 0} for modality in modalities}

for modality in modalities:
    for fold in folds:
        base_path = f'/UK_BB/brainbody/lifestyle'
        folds_path = os.path.join(base_path, f'folds/fold_{fold}')
        suppl_path = os.path.join(folds_path, 'suppl')
        scaling_path = os.path.join(folds_path, 'scaling')
        models_path = os.path.join(folds_path, 'models')
        g_pred_path = os.path.join(folds_path, 'g_pred')
        
        os.makedirs(folds_path, exist_ok=True)
        os.makedirs(suppl_path, exist_ok=True)
        os.makedirs(scaling_path, exist_ok=True)
        os.makedirs(models_path, exist_ok=True)
        os.makedirs(g_pred_path, exist_ok=True)

        g_train = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_train_with_id_{fold}.csv')
        g_test = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_test_with_id_{fold}.csv')
        features = pd.read_csv(os.path.join(base_path, f'data/{modality}.csv'))
        print('Features shape', features.shape)

        feature_columns = features.drop(columns='eid').columns

        # Merge features with targets
        train_merge_all = pd.merge(features, g_train, on='eid').reset_index(drop=True)
        test_merge_all = pd.merge(features, g_test, on='eid').reset_index(drop=True)

        # Save merged data
        train_merge_all.to_csv(os.path.join(suppl_path, f'{modality}_train_feat_targ_fold_{fold}.csv'), index=False)
        test_merge_all.to_csv(os.path.join(suppl_path, f'{modality}_test_feat_targ_fold_{fold}.csv'), index=False)

        print(f'==== Train shape ====\n {modality} - Fold {fold}', train_merge_all.shape)
        print(f'==== Test shape ====\n {modality} - Fold {fold}', test_merge_all.shape)

        # Update the number of observations for the current modality
        modality_observations[modality]['train'] += train_merge_all.shape[0]
        modality_observations[modality]['test'] += test_merge_all.shape[0]

        # Extract features and save
        train_merge_all[feature_columns].to_csv(os.path.join(suppl_path, f'{modality}_train_fold_{fold}.csv'), index=False)
        test_merge_all[feature_columns].to_csv(os.path.join(suppl_path, f'{modality}_test_fold_{fold}.csv'), index=False)

        # Scale features
        print('Scaling')
        scaler_features = StandardScaler()
        features_train_scaled = scaler_features.fit_transform(train_merge_all[feature_columns])
        features_test_scaled = scaler_features.transform(test_merge_all[feature_columns])
        pd.DataFrame(features_train_scaled, columns=feature_columns).to_csv(os.path.join(scaling_path, f'{modality}_train_scaled_fold_{fold}.csv'), index=False)
        pd.DataFrame(features_test_scaled, columns=feature_columns).to_csv(os.path.join(scaling_path, f'{modality}_test_scaled_fold_{fold}.csv'), index=False)

        # Save scaler
        with open(os.path.join(scaling_path, f'{modality}_scaler_features_fold_{fold}.pkl'), "wb") as f:
            pickle.dump(scaler_features, f)

best_modality = max(modality_observations, key=lambda x: modality_observations[x]['train'] + modality_observations[x]['test'])
total_observations = modality_observations[best_modality]['train'] + modality_observations[best_modality]['test']

print('======================================================================================================')
print(f'The modality with the highest number of observations is: {best_modality}: n = {total_observations}')
print(f'Observations in each modality train/test sets:\n', modality_observations)

# Sexual factors

In [ ]:
df_sex_factors = ukbiobank.utils.utils.loadCsv(ukbio=ukb, fields=['eid',
#2129, #Answered sexual history questions
2139, #Age first had sexual intercourse
2149, #Lifetime number of sexual partners
2159, #Ever had same-sex intercourse
3669 #Lifetime number of same-sex sexual partners
])
df_sex_factors_names = ukbiobank.utils.utils.fieldIdsToNames(ukbio=ukb, df=df_sex_factors)
df_sex_factors_names.to_csv(os.path.join(data_path, 'sexual_factors_vars.csv'), index=False)

In [ ]:
print('% missing')
with pd.option_context('display.max_rows', None):
    display(((df_sex_factors_names.isna().sum() / len(df_sex_factors_names)).round(2)*100).sort_values(ascending=False))

In [ ]:
# Filter Instance 2
df_sex_factors_names = pd.read_csv(os.path.join(data_path, 'sexual_factors_vars.csv'))
sex_fact_i2 = df_sex_factors_names[['eid'] + df_sex_factors_names.filter(regex=r'2\.\d$').columns.tolist()]
sex_fact_i2.columns = sex_fact_i2.columns.str.replace('-2.0', '')
print('SHAPE:', sex_fact_i2.shape)
with pd.option_context('display.max_rows', None):
    display(((sex_fact_i2.isna().sum() / len(sex_fact_i2)).round(2)*100).sort_values(ascending=False))

In [ ]:
print('MIN\n', sex_fact_i2.min())
print('Number of -1 responses\n', (sex_fact_i2 == -1).sum().sort_values(ascending=False))
print('Number of -2 responses\n', (sex_fact_i2 == -2).sum().sort_values(ascending=False))
print('Number of -3 responses\n', (sex_fact_i2 == -3).sum().sort_values(ascending=False))
print('Number of -10 responses\n', (sex_fact_i2 == -10).sum().sort_values(ascending=False))

Collected from all participants except those who indicated they never had had sexual intercourse:
- Lifetime number of sexual partners
- Ever had same-sex intercourse

Collected from participants those who indicated they had had sexual intercourse with someone of the same sex:
- 'Lifetime number of same-sex sexual partners'

In [ ]:
# Fill NA
sex_fact_i2 = sex_fact_i2.dropna(subset='Age first had sexual intercourse').reset_index(drop=True)

sex_intercourse = ['Lifetime number of sexual partners', 'Ever had same-sex intercourse']
sex_fact_i2.loc[sex_fact_i2['Age first had sexual intercourse'] == -2, sex_intercourse] = sex_fact_i2.loc[
        sex_fact_i2['Age first had sexual intercourse'] == -2, sex_intercourse].fillna(0)

#-2 = Never had sex

sex_fact_i2.loc[sex_fact_i2['Ever had same-sex intercourse'] == 0, 'Lifetime number of same-sex sexual partners'] = sex_fact_i2.loc[
        sex_fact_i2['Ever had same-sex intercourse'] == 0, 'Lifetime number of same-sex sexual partners'].fillna(0)

print('SHAPE:', sex_fact_i2.shape)
print('NAs:', (sex_fact_i2.isna().sum() / len(sex_fact_i2)).round(2))
print('MIN\n', sex_fact_i2.min())
print('MAX\n', sex_fact_i2.max())

In [ ]:
sex_fact_i2_encode = sex_fact_i2.copy()

In [ ]:
# Define the mapping for categorical responses
sex_fact_mapping = {
    -2: 'Never had sex',
    -1: 'Do not know / Prefer not to answer',
    -3: 'Do not know / Prefer not to answer'
}

# Create label column (automatically converts unmapped values to NaN)
sex_fact_i2_encode['Age first had sexual intercourse - Label'] = (
    sex_fact_i2_encode['Age first had sexual intercourse']
    .map(sex_fact_mapping)
)

# One-hot encoding only for categorical responses
dummies = pd.get_dummies(
    sex_fact_i2_encode['Age first had sexual intercourse - Label'], 
    prefix='Age first had sexual intercourse'
)

# Clean column names
dummies.columns = [col.replace("_", " - ") for col in dummies.columns]

# Combine with original data
sex_fact_i2_encode = pd.concat([
    sex_fact_i2_encode,
    dummies
], axis=1)

# Optional: Drop the intermediate label column if needed
sex_fact_i2_encode.drop(columns=['Age first had sexual intercourse - Label'], inplace=True)

!!! Important. Only in this continous variable (age) I assign non-responders to a new column, because there was an option 'never had sex' coded as -2, which was also dummy-encoded.

In [ ]:
# Assign NAs to non-responders
sex_fact_i2_encode.replace(-1, np.nan, inplace=True)
sex_fact_i2_encode.replace(-2, np.nan, inplace=True)
sex_fact_i2_encode.replace(-3, np.nan, inplace=True)

print('NA\n', (sex_fact_i2_encode.isna()).sum().sort_values(ascending=False))
print('MIN\n', sex_fact_i2_encode.min())
print('NEG\n', (sex_fact_i2_encode < 0).sum().sort_values(ascending=False))

sex_fact_i2_encode.to_csv(os.path.join(data_path, 'sexual_factors.csv'), index=False)

In [ ]:
# Check if there are columns with constant values (e.g., all 0)
def is_constant(column):
    """Check if all values in a column are the same."""
    return np.all(column == column.iloc[0])

def find_constant_columns(df, df_name):
    constant_columns = []
    
    print(f"\nChecking for constant columns in {df_name}...")
    
    for column in df.columns:
        if is_constant(df[column]):
            constant_columns.append(column)
            print(f"✅ Column '{column}' is constant.")
        else:
            print(f"❌ Column '{column}' is NOT constant.")
    
    if constant_columns:
        print(f"\n🔍 Found {len(constant_columns)} constant column(s) in {df_name}: {constant_columns}")
    else:
        print(f"\n🎉 No constant columns found in {df_name}!")
    
    return constant_columns

find_constant_columns(sex_fact_i2_encode, "sex_fact_i2_encode")

In [ ]:
# Match features to the target and compute sample sizes
folds = range(0,5)
modalities = ['sexual_factors']

modality_observations = {modality: {'train': 0, 'test': 0} for modality in modalities}

for modality in modalities:
    for fold in folds:
        base_path = f'/UK_BB/brainbody/lifestyle'
        folds_path = os.path.join(base_path, f'folds/fold_{fold}')
        suppl_path = os.path.join(folds_path, 'suppl')
        scaling_path = os.path.join(folds_path, 'scaling')
        models_path = os.path.join(folds_path, 'models')
        g_pred_path = os.path.join(folds_path, 'g_pred')
        
        os.makedirs(folds_path, exist_ok=True)
        os.makedirs(suppl_path, exist_ok=True)
        os.makedirs(scaling_path, exist_ok=True)
        os.makedirs(models_path, exist_ok=True)
        os.makedirs(g_pred_path, exist_ok=True)

        g_train = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_train_with_id_{fold}.csv')
        g_test = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_test_with_id_{fold}.csv')
        features = pd.read_csv(os.path.join(base_path, f'data/{modality}.csv'))
        print('Features shape', features.shape)

        feature_columns = features.drop(columns='eid').columns

        # Merge features with targets
        train_merge_all = pd.merge(features, g_train, on='eid').reset_index(drop=True)
        test_merge_all = pd.merge(features, g_test, on='eid').reset_index(drop=True)

        # Save merged data
        train_merge_all.to_csv(os.path.join(suppl_path, f'{modality}_train_feat_targ_fold_{fold}.csv'), index=False)
        test_merge_all.to_csv(os.path.join(suppl_path, f'{modality}_test_feat_targ_fold_{fold}.csv'), index=False)

        print(f'==== Train shape ====\n {modality} - Fold {fold}', train_merge_all.shape)
        print(f'==== Test shape ====\n {modality} - Fold {fold}', test_merge_all.shape)

        # Update the number of observations for the current modality
        modality_observations[modality]['train'] += train_merge_all.shape[0]
        modality_observations[modality]['test'] += test_merge_all.shape[0]

        # Extract features and save
        train_merge_all[feature_columns].to_csv(os.path.join(suppl_path, f'{modality}_train_fold_{fold}.csv'), index=False)
        test_merge_all[feature_columns].to_csv(os.path.join(suppl_path, f'{modality}_test_fold_{fold}.csv'), index=False)

        # Scale features
        print('Scaling')
        scaler_features = StandardScaler()
        features_train_scaled = scaler_features.fit_transform(train_merge_all[feature_columns])
        features_test_scaled = scaler_features.transform(test_merge_all[feature_columns])
        pd.DataFrame(features_train_scaled, columns=feature_columns).to_csv(os.path.join(scaling_path, f'{modality}_train_scaled_fold_{fold}.csv'), index=False)
        pd.DataFrame(features_test_scaled, columns=feature_columns).to_csv(os.path.join(scaling_path, f'{modality}_test_scaled_fold_{fold}.csv'), index=False)

        # Save scaler
        with open(os.path.join(scaling_path, f'{modality}_scaler_features_fold_{fold}.pkl'), "wb") as f:
            pickle.dump(scaler_features, f)

best_modality = max(modality_observations, key=lambda x: modality_observations[x]['train'] + modality_observations[x]['test'])
total_observations = modality_observations[best_modality]['train'] + modality_observations[best_modality]['test']

print('======================================================================================================')
print(f'The modality with the highest number of observations is: {best_modality}: n = {total_observations}')
print(f'Observations in each modality train/test sets:\n', modality_observations)

# Smoking

In [ ]:
df_smoking = ukbiobank.utils.utils.loadCsv(ukbio=ukb, fields=['eid',
20160, #Ever smoked
20162, #Pack years adult smoking as proportion of life span exposed to smoking
20161, #Pack years of smoking
#10895, #Light smokers, at least 100 smokes in lifetime (pilot)
20116, #Smoking status
1239, #Current tobacco smoking
1249, #Past tobacco smoking
2644, #Light smokers, at least 100 smokes in lifetime
3436, #Age started smoking in current smokers
3446, #Type of tobacco currently smoked
5959, #Previously smoked cigarettes on most/all days
3456, #Number of cigarettes currently smoked daily (current cigarette smokers)
6194, #Age stopped smoking cigarettes (current cigar/pipe or previous cigarette smoker)
6183, #Number of cigarettes previously smoked daily (current cigar/pipe smokers)
3466, #Time from waking to first cigarette
3476, #Difficulty not smoking for 1 day
3486, #Ever tried to stop smoking
3496, #Wants to stop smoking
3506, #Smoking compared to 10 years previous
6158, #Why reduced smoking
2867, #Age started smoking in former smokers
2877, #Type of tobacco previously smoked
2887, #Number of cigarettes previously smoked daily
2897, #Age stopped smoking
2907, #Ever stopped smoking for 6+ months
#10827, #Ever stopped smoking for 6+ months (pilot)
6157, #Why stopped smoking
#10115, #Why stopped smoking (pilot)
2926, #Number of unsuccessful stop-smoking attempts
2936, #Likelihood of resuming smoking
1259, #Smoking/smokers in household
1269, #Exposure to tobacco smoke at home
1279, #Exposure to tobacco smoke outside home
])
df_smoking_names = ukbiobank.utils.utils.fieldIdsToNames(ukbio=ukb, df=df_smoking)
df_smoking_names.to_csv(os.path.join(data_path, 'smoking_vars.csv'), index=False)

In [ ]:
print('% missing')
with pd.option_context('display.max_rows', None):
    display(((df_smoking_names.isna().sum() / len(df_smoking_names)).round(2)*100).sort_values(ascending=False))

In [ ]:
# Filter Instance 2
df_smoking_names = pd.read_csv(os.path.join(data_path, 'smoking_vars.csv'))
smoking_i2 = df_smoking_names[['eid'] + df_smoking_names.filter(regex=r'2\.\d$').columns.tolist()]
#smoking_i2.columns = smoking_i2.columns.str.replace('-2.0', '')
print('SHAPE:', smoking_i2.shape)
with pd.option_context('display.max_rows', None):
    display(((smoking_i2.isna().sum() / len(smoking_i2)).round(2)*100).sort_values(ascending=False))

In [ ]:
print('MIN\n', smoking_i2.min())
print('Number of -1 responses\n', (smoking_i2 == -1).sum().sort_values(ascending=False))
print('Number of -2 responses\n', (smoking_i2 == -2).sum().sort_values(ascending=False))
print('Number of -3 responses\n', (smoking_i2 == -3).sum().sort_values(ascending=False))
print('Number of -10 responses\n', (smoking_i2 == -10).sum().sort_values(ascending=False))

In [ ]:
smoking_i2 = smoking_i2.dropna(subset=['Ever smoked-2.0']).reset_index(drop=True)
print('SHAPE:', smoking_i2.shape)
with pd.option_context('display.max_rows', None):
    display(smoking_i2.isna().sum().sort_values(ascending=False))

In [ ]:
# Replace -10  with 0.5
smoking_i2.replace({-10: 0.5, #Less than 1
}, inplace=True)
print('Number of -10 responses in Instance 0\n', (smoking_i2 == -10).sum().sort_values(ascending=False))

#### Fill NA

In [ ]:
smoking_i2_encoding = smoking_i2.copy()

In [ ]:
# Past tobacco smokings
smoking_i2_encoding['Current tobacco smoking-2.0'] = smoking_i2_encoding['Current tobacco smoking-2.0'].astype(int)

# Fill NaN values in 'Past tobacco smoking-2.0' where 'Current tobacco smoking-2.0' is 1
smoking_i2_encoding.loc[
    smoking_i2_encoding['Current tobacco smoking-2.0'] == 1, 'Past tobacco smoking-2.0'] = smoking_i2_encoding.loc[
        smoking_i2_encoding['Current tobacco smoking-2.0'] == 1, 'Past tobacco smoking-2.0'].fillna(0)

# Verify if NaN values are filled
print(smoking_i2_encoding['Past tobacco smoking-2.0'].isna().sum())

In [ ]:
# Smoking/smokers in household
smoking_i2_encoding.loc[
    smoking_i2_encoding['Current tobacco smoking-2.0'] == 1, 'Smoking/smokers in household-2.0'] = smoking_i2_encoding.loc[
        smoking_i2_encoding['Current tobacco smoking-2.0'] == 1, 'Smoking/smokers in household-2.0'].fillna(0)
# Exposure to tobacco smoke at home
smoking_i2_encoding.loc[
    smoking_i2_encoding['Current tobacco smoking-2.0'] == 1, 'Exposure to tobacco smoke at home-2.0'] = smoking_i2_encoding.loc[
        smoking_i2_encoding['Current tobacco smoking-2.0'] == 1, 'Exposure to tobacco smoke at home-2.0'].fillna(0)
# Exposure to tobacco smoke outside home
smoking_i2_encoding.loc[
    smoking_i2_encoding['Current tobacco smoking-2.0'] == 1, 'Exposure to tobacco smoke outside home-2.0'] = smoking_i2_encoding.loc[
        smoking_i2_encoding['Current tobacco smoking-2.0'] == 1, 'Exposure to tobacco smoke outside home-2.0'].fillna(0)

print(smoking_i2_encoding['Smoking/smokers in household-2.0'].isna().sum())
print(smoking_i2_encoding['Smoking/smokers in household-2.0'].isna().sum())
print(smoking_i2_encoding['Smoking/smokers in household-2.0'].isna().sum())

In [ ]:
# Ever smoked
never_smoked = smoking_i2_encoding.drop(columns=['eid', 'Ever smoked-2.0', 'Pack years of smoking-2.0']).columns.to_list()
smoking_i2_encoding.loc[
    smoking_i2_encoding['Ever smoked-2.0'] != 1, never_smoked] = smoking_i2_encoding.loc[
        smoking_i2_encoding['Ever smoked-2.0'] != 1, never_smoked].fillna(0)

print(smoking_i2_encoding['Ever smoked-2.0'].isna().sum())

In [ ]:
# Light smokers, at least 100 smokes in lifetime
smoking_i2_encoding['Past tobacco smoking-2.0'] = smoking_i2_encoding['Past tobacco smoking-2.0'].astype(int)

smoking_i2_encoding.loc[
    (smoking_i2_encoding['Past tobacco smoking-2.0'] != 2) & 
    (smoking_i2_encoding['Past tobacco smoking-2.0'] != 3), 
    'Light smokers, at least 100 smokes in lifetime-2.0'
] = smoking_i2_encoding.loc[
    (smoking_i2_encoding['Past tobacco smoking-2.0'] != 2) & 
    (smoking_i2_encoding['Past tobacco smoking-2.0'] != 3), 
    'Light smokers, at least 100 smokes in lifetime-2.0'
].fillna(0)
print(smoking_i2_encoding['Light smokers, at least 100 smokes in lifetime-2.0'].isna().sum())

In [ ]:
# Age started smoking in current smokers
smoking_i2_encoding.loc[
    (smoking_i2_encoding['Current tobacco smoking-2.0'] != 1), 
    'Age started smoking in current smokers-2.0'] = smoking_i2_encoding.loc[
    (smoking_i2_encoding['Current tobacco smoking-2.0'] != 1),
    'Age started smoking in current smokers-2.0'].fillna(0)

print(smoking_i2_encoding['Age started smoking in current smokers-2.0'].isna().sum())

In [ ]:
# Type of tobacco currently smoked
smoking_i2_encoding.loc[
    smoking_i2_encoding['Current tobacco smoking-2.0'] != 1, 'Type of tobacco currently smoked-2.0'] = smoking_i2_encoding.loc[
        smoking_i2_encoding['Current tobacco smoking-2.0'] != 1, 'Type of tobacco currently smoked-2.0'].fillna(0)

print(smoking_i2_encoding['Type of tobacco currently smoked-2.0'].isna().sum())

In [ ]:
# Previously smoked cigarettes on most/all days
smoking_i2_encoding['Type of tobacco currently smoked-2.0'] = smoking_i2_encoding['Type of tobacco currently smoked-2.0'].astype(int)

smoking_i2_encoding.loc[
    (smoking_i2_encoding['Current tobacco smoking-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 3), 
    'Previously smoked cigarettes on most/all days-2.0'] = smoking_i2_encoding.loc[
    (smoking_i2_encoding['Current tobacco smoking-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 3),
    'Previously smoked cigarettes on most/all days-2.0'].fillna(0)

print(smoking_i2_encoding['Previously smoked cigarettes on most/all days-2.0'].isna().sum())

In [ ]:
smoking_i2_encoding['Type of tobacco currently smoked-2.0'] = smoking_i2_encoding['Type of tobacco currently smoked-2.0'].astype(int)
smoking_i2_encoding['Type of tobacco currently smoked-2.0'] = smoking_i2_encoding['Type of tobacco currently smoked-2.0'].astype(int)


# Fill NaN values in 'Previously smoked cigarettes on most/all days-2.0' where conditions are met
smoking_i2_encoding['Previously smoked cigarettes on most/all days-2.0'] = np.where(
    (smoking_i2_encoding['Current tobacco smoking-2.0'] != 1) & 
    (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 3) & 
    (smoking_i2_encoding['Previously smoked cigarettes on most/all days-2.0'].isna()), 
    0, 
    smoking_i2_encoding['Previously smoked cigarettes on most/all days-2.0']
)

# Verify if NaN values are filled
print(smoking_i2_encoding['Previously smoked cigarettes on most/all days-2.0'].isna().sum())

In [ ]:
# Number of cigarettes currently smoked daily (current cigarette smokers)
col_name = 'Number of cigarettes currently smoked daily (current cigarette smokers)-2.0'

# Condition: Either not current smoker OR smokes non-cigarette tobacco
condition = (
    (smoking_i2_encoding['Current tobacco smoking-2.0'] != 1) | 
    (~smoking_i2_encoding['Type of tobacco currently smoked-2.0'].isin([1, 2]))
)

# Apply the fillna(0) only to rows meeting condition
smoking_i2_encoding.loc[condition, col_name] = (
    smoking_i2_encoding.loc[condition, col_name].fillna(0))

In [ ]:
# Age stopped smoking cigarettes (current cigar/pipe or previous cigarette smoker)
'''Field 6194 was collected from participants who indicated they currently smoke cigars and pipes,
as defined by their answers to Field 3446 and previously smoked cigarettes on most or all days, as defined by their answers to Field 3446
'''
smoking_i2_encoding['Previously smoked cigarettes on most/all days-2.0'] = smoking_i2_encoding['Previously smoked cigarettes on most/all days-2.0'].astype(int)
smoking_i2_encoding['Type of tobacco currently smoked-2.0'] = smoking_i2_encoding['Type of tobacco currently smoked-2.0'].astype(int)

smoking_i2_encoding.loc[
    (smoking_i2_encoding['Previously smoked cigarettes on most/all days-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 3), 
    'Age stopped smoking cigarettes (current cigar/pipe or previous cigarette smoker)-2.0'
] = smoking_i2_encoding.loc[
    (smoking_i2_encoding['Previously smoked cigarettes on most/all days-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 3),
    'Age stopped smoking cigarettes (current cigar/pipe or previous cigarette smoker)-2.0'].fillna(0)

print(smoking_i2_encoding['Age stopped smoking cigarettes (current cigar/pipe or previous cigarette smoker)-2.0'].isna().sum())

In [ ]:
# Time from waking to first cigarette
smoking_i2_encoding.loc[
    (smoking_i2_encoding['Current tobacco smoking-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 2), 
    'Time from waking to first cigarette-2.0'] = smoking_i2_encoding.loc[
    (smoking_i2_encoding['Current tobacco smoking-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 2),
    'Time from waking to first cigarette-2.0'].fillna(0)


# Difficulty not smoking for 1 day
smoking_i2_encoding.loc[
    (smoking_i2_encoding['Current tobacco smoking-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 2), 
    'Difficulty not smoking for 1 day-2.0'] = smoking_i2_encoding.loc[
    (smoking_i2_encoding['Current tobacco smoking-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 2),
    'Difficulty not smoking for 1 day-2.0'].fillna(0)

In [ ]:
# Ever tried to stop smoking
smoking_i2_encoding.loc[(smoking_i2_encoding['Current tobacco smoking-2.0'] != 1), 'Ever tried to stop smoking-2.0'] = smoking_i2_encoding.loc[
(smoking_i2_encoding['Current tobacco smoking-2.0'] != 1), 'Ever tried to stop smoking-2.0'].fillna(0)

# Wants to stop smoking
smoking_i2_encoding.loc[(smoking_i2_encoding['Current tobacco smoking-2.0'] != 1), 'Wants to stop smoking-2.0'] = smoking_i2_encoding.loc[
    (smoking_i2_encoding['Current tobacco smoking-2.0'] != 1),'Wants to stop smoking-2.0'].fillna(0)

# Smoking compared to 10 years previous
smoking_i2_encoding.loc[
    (smoking_i2_encoding['Current tobacco smoking-2.0'] != 1), 
    'Smoking compared to 10 years previous-2.0'] = smoking_i2_encoding.loc[
    (smoking_i2_encoding['Current tobacco smoking-2.0'] != 1),
    'Smoking compared to 10 years previous-2.0'].fillna(0)

In [ ]:
# Why reduced smoking
why_reduced_smoking = ['Why reduced smoking-2.0', 'Why reduced smoking-2.1', 'Why reduced smoking-2.2', 'Why reduced smoking-2.3']
smoking_i2_encoding.loc[
    (smoking_i2_encoding['Current tobacco smoking-2.0'] != 1) & (smoking_i2_encoding['Smoking compared to 10 years previous-2.0'] != 3), 
    why_reduced_smoking] = smoking_i2_encoding.loc[
    (smoking_i2_encoding['Current tobacco smoking-2.0'] != 1) & (smoking_i2_encoding['Smoking compared to 10 years previous-2.0'] != 3),
    why_reduced_smoking].fillna(0)

In [ ]:
# Age started smoking in former smokers
smoking_i2_encoding.loc[
    smoking_i2_encoding['Past tobacco smoking-2.0'] != 1, 'Age started smoking in former smokers-2.0'] = smoking_i2_encoding.loc[
        smoking_i2_encoding['Past tobacco smoking-2.0'] != 1, 'Age started smoking in former smokers-2.0'].fillna(0)
#in the past they smoked on most or all days

In [ ]:
# Type of tobacco previously smoked
smoking_i2_encoding.loc[
    smoking_i2_encoding['Past tobacco smoking-2.0'] != 1, 'Type of tobacco previously smoked-2.0'] = smoking_i2_encoding.loc[
        smoking_i2_encoding['Past tobacco smoking-2.0'] != 1, 'Type of tobacco previously smoked-2.0'].fillna(0)

In [ ]:
# Number of cigarettes previously smoked daily (current cigar/pipe smokers)
'''Field 6183 was collected from participants who indicated they
currently smoke cigars and pipes, as defined by their answers to Field 3446 and
previously smoked cigarettes on most or all days, as defined by their answers to Field 3446
'''
smoking_i2_encoding.loc[
    (smoking_i2_encoding['Previously smoked cigarettes on most/all days-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 3), 
    'Number of cigarettes previously smoked daily-2.0'
] = smoking_i2_encoding.loc[
    (smoking_i2_encoding['Previously smoked cigarettes on most/all days-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 3),
    'Number of cigarettes previously smoked daily-2.0'].fillna(0)

print(smoking_i2_encoding['Number of cigarettes previously smoked daily-2.0'].isna().sum())

In [ ]:
# Age stopped smoking
smoking_i2_encoding.loc[
    smoking_i2_encoding['Past tobacco smoking-2.0'] != 1, 'Age stopped smoking-2.0'] = smoking_i2_encoding.loc[
        smoking_i2_encoding['Past tobacco smoking-2.0'] != 1, 'Age stopped smoking-2.0'].fillna(0)

smoking_i2_encoding.loc[
    (smoking_i2_encoding['Previously smoked cigarettes on most/all days-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 3), 
    'Number of cigarettes previously smoked daily-2.0'
] = smoking_i2_encoding.loc[
    (smoking_i2_encoding['Previously smoked cigarettes on most/all days-2.0'] != 1) & (smoking_i2_encoding['Type of tobacco currently smoked-2.0'] != 3),
    'Number of cigarettes previously smoked daily-2.0'].fillna(0)

In [ ]:
# Ever stopped smoking for 6+ months
smoking_i2_encoding.loc[
    smoking_i2_encoding['Past tobacco smoking-2.0'] != 1, 'Ever stopped smoking for 6+ months-2.0'] = smoking_i2_encoding.loc[
        smoking_i2_encoding['Past tobacco smoking-2.0'] != 1, 'Ever stopped smoking for 6+ months-2.0'].fillna(0)

In [ ]:
# Why stopped smoking
why_stopped_smoking = ['Why stopped smoking-2.0', 'Why stopped smoking-2.1', 'Why stopped smoking-2.2', 'Why stopped smoking-2.3']
smoking_i2_encoding.loc[
    (smoking_i2_encoding['Past tobacco smoking-2.0'] != 1) & (smoking_i2_encoding['Ever stopped smoking for 6+ months-2.0'] != 1), 
    why_stopped_smoking] = smoking_i2_encoding.loc[
    (smoking_i2_encoding['Past tobacco smoking-2.0'] != 1) & (smoking_i2_encoding['Ever stopped smoking for 6+ months-2.0'] != 1),
    why_stopped_smoking].fillna(0)

In [ ]:
# Number of unsuccessful stop-smoking attempts
smoking_i2_encoding.loc[
    (smoking_i2_encoding['Past tobacco smoking-2.0'] != 1) & (smoking_i2_encoding['Ever stopped smoking for 6+ months-2.0'] != 1), 
    'Number of unsuccessful stop-smoking attempts-2.0'] = smoking_i2_encoding.loc[
    (smoking_i2_encoding['Past tobacco smoking-2.0'] != 1) & (smoking_i2_encoding['Ever stopped smoking for 6+ months-2.0'] != 1),
    'Number of unsuccessful stop-smoking attempts-2.0'].fillna(0)

In [ ]:
# Likelihood of resuming smoking
smoking_i2_encoding.loc[
    (smoking_i2_encoding['Past tobacco smoking-2.0'] != 1) & (smoking_i2_encoding['Ever stopped smoking for 6+ months-2.0'] != 1), 
    'Likelihood of resuming smoking-2.0'] = smoking_i2_encoding.loc[
    (smoking_i2_encoding['Past tobacco smoking-2.0'] != 1) & (smoking_i2_encoding['Ever stopped smoking for 6+ months-2.0'] != 1),
    'Likelihood of resuming smoking-2.0'].fillna(0)

In [ ]:
print('SHAPE:', smoking_i2_encoding.shape)
print('NAs:', smoking_i2_encoding.isna().sum().sort_values(ascending=False))

#### One-hot encode

In [ ]:
smoking_i2_dummy = smoking_i2_encoding.copy()
print('SHAPE:', smoking_i2_dummy.shape)
print('NAs:', smoking_i2_dummy.isna().sum().sort_values(ascending=False))

In [ ]:
# Validation function
def process_categorical_column(df, col_name, mapping, prefix=None):
    """
    Processes a categorical column, mapping, and One-hot encoding
    """
    if prefix is None:
        prefix = col_name
    
    valid_values = set(mapping.keys())
    
    # Validation
    invalid_mask = ~df[col_name].apply(lambda x: is_valid_value(x, valid_values)) & df[col_name].notna()
    if invalid_mask.any():
        invalid_values = df.loc[invalid_mask, col_name].unique()
        raise ValueError(f"Unexpected values in '{col_name}': {invalid_values}. Update mapping or clean data.")
    
    # Mapping
    df[col_name] = df[col_name].astype(float).map(mapping)
    
    # One-hot encoding
    dummies = pd.get_dummies(df[col_name], prefix=prefix).astype(int)
    dummies.columns = [col.replace("_", " - ") for col in dummies.columns]
    
    # Return processed dataframe
    return pd.concat([df.drop(columns=col_name), dummies], axis=1)

# Helper function
def is_valid_value(x, valid_values):
    if pd.isna(x):
        return True
    try:
        return float(x) in valid_values
    except (ValueError, TypeError):
        return False

In [ ]:
# Why reduced smoking
why_reduced_smoking = ['Why reduced smoking-2.0', 'Why reduced smoking-2.1',
                      'Why reduced smoking-2.2', 'Why reduced smoking-2.3']

why_reduced_mapping = {
    0: 'Did not reduce smoking',
    1: 'Illness or ill health',
    2: "Doctor's advice",
    3: 'Health precaution',
    4: 'Financial reasons',
    -7: 'Other reason',
    -1: 'Do not know / Prefer not to answer',
    -3: 'Do not know / Prefer not to answer'
}

# Add this validation right after defining why_reduced_mapping
valid_values = set(why_reduced_mapping.keys())
invalid_mask = smoking_i2_dummy[why_reduced_smoking].apply(lambda col: ~col.isin(valid_values) & col.notna())
if invalid_mask.any().any():
    invalid = smoking_i2_dummy[why_reduced_smoking][invalid_mask].stack().unique()
    raise ValueError(f"Unexpected values: {invalid}. Update mapping or clean data.")

# Convert with NA protection
smoking_i2_dummy[why_reduced_smoking] = smoking_i2_dummy[why_reduced_smoking].apply(
    lambda col: pd.to_numeric(col, errors='coerce').map(why_reduced_mapping))

# Create dummies
stacked = smoking_i2_dummy[why_reduced_smoking].stack()
dummies = pd.get_dummies(stacked, prefix='Why reduced smoking')

# Clean processing
dummies = dummies.groupby(level=0).max()  # Better than sum() for multi-select
dummies.columns = [col.replace("_", " - ") for col in dummies.columns]

# Final merge
smoking_i2_dummy = pd.concat([smoking_i2_dummy, dummies], axis=1)
smoking_i2_dummy = smoking_i2_dummy.drop(columns=why_reduced_smoking)

print('SHAPE:', smoking_i2_dummy.shape)
print('NAs:', smoking_i2_dummy.isna().sum().sort_values(ascending=False))

In [ ]:
# Why stopped smoking
why_stopped_smoking = ['Why stopped smoking-2.0', 'Why stopped smoking-2.1',
                      'Why stopped smoking-2.2', 'Why stopped smoking-2.3']

why_stopped_mapping = {
    0: 'Did not stop smoking',
    1: 'Illness or ill health',
    2: "Doctor's advice",
    3: 'Health precaution',
    4: 'Financial reasons',
    -7: 'Other reason',
    -1: 'Do not know / Prefer not to answer',
    -3: 'Do not know / Prefer not to answer'
}

# Add this validation right after defining why_stopped_mapping
valid_values = set(why_stopped_mapping.keys())
invalid_mask = smoking_i2_dummy[why_stopped_smoking].apply(lambda col: ~col.isin(valid_values) & col.notna())
if invalid_mask.any().any():
    invalid = smoking_i2_dummy[why_stopped_smoking][invalid_mask].stack().unique()
    raise ValueError(f"Unexpected values: {invalid}. Update mapping or clean data.")

# Convert with NA protection
smoking_i2_dummy[why_stopped_smoking] = smoking_i2_dummy[why_stopped_smoking].apply(
    lambda col: pd.to_numeric(col, errors='coerce').map(why_stopped_mapping))

# Create dummies
stacked = smoking_i2_dummy[why_stopped_smoking].stack()
dummies = pd.get_dummies(stacked, prefix='Why stopped smoking')

# Clean processing
dummies = dummies.groupby(level=0).max()  # Better than sum() for multi-select
dummies.columns = [col.replace("_", " - ") for col in dummies.columns]

# Final merge
smoking_i2_dummy = pd.concat([smoking_i2_dummy, dummies], axis=1)
smoking_i2_dummy = smoking_i2_dummy.drop(columns=why_stopped_smoking)

print('SHAPE:', smoking_i2_dummy.shape)
print('NAs:', smoking_i2_dummy.isna().sum().sort_values(ascending=False))

In [ ]:
'''# Why stopped smoking
why_stopped_smoking = ['Why stopped smoking-2.0', 'Why stopped smoking-2.1', 'Why stopped smoking-2.2', 'Why stopped smoking-2.3']
why_stopped_mapping = {
    1: 'Illness or ill health',
    2: "Doctor's advice",
    3: 'Health precaution',
    4: 'Financial reasons',
    -7: 'Other reason',
    -1: 'Do not know / Prefer not to answer',
    -3: 'Do not know / Prefer not to answer'
}

smoking_i2_dummy[why_stopped_smoking] = smoking_i2_dummy[why_stopped_smoking].apply(
    lambda col: col.map(why_stopped_mapping))

stacked = smoking_i2_dummy[why_stopped_smoking].stack()  
dummies = pd.get_dummies(stacked, prefix='Why stopped smoking')
dummies = dummies.groupby(level=0).sum()
dummies = dummies.reindex(smoking_i2_dummy.index, fill_value=0)
dummies.columns = [col.replace("_", " - ") for col in dummies.columns]
smoking_i2_dummy = pd.concat([smoking_i2_dummy, dummies], axis=1)
smoking_i2_dummy = smoking_i2_dummy.drop(columns=why_stopped_smoking)
print('SHAPE:', smoking_i2_dummy.shape)
print('NAs:', ((smoking_i2_dummy.isna().sum() / len(smoking_i2_dummy)).round(2)*100).sort_values(ascending=False))'
'''

In [ ]:
# Smoking status
smoking_status_mapping = {
-3: 'Prefer not to answer',
0: 'Never',
1: 'Previous',
2: 'Current'
}

smoking_i2_dummy['Smoking status-2.0'] = (smoking_i2_dummy['Smoking status-2.0'].map(smoking_status_mapping))

dummies = pd.get_dummies(smoking_i2_dummy['Smoking status-2.0'], prefix='Smoking status').astype(int)

dummies.columns = [col.replace("_", " - ") for col in dummies.columns]
smoking_i2_dummy = smoking_i2_dummy.drop(columns='Smoking status-2.0')
smoking_i2_dummy = pd.concat([smoking_i2_dummy, dummies], axis=1)

In [ ]:
# Current tobacco smoking
smoking_i2_dummy.loc[:, 'Current tobacco smoking-2.0'] = smoking_i2_dummy.loc[:, 'Current tobacco smoking-2.0'].replace({
1: 2, # was: 1=Yes, on most or all days, become: 1=Only occasionally
2: 1 # was: 1=Only occasionally, become: 1=Yes, on most or all days
})

# Original
#1 Yes, on most or all days
#2  Only occasionally
#0 No
#-3 Prefer not to answer

# Desired
#2 Yes, on most or all days
#1 Only occasionally 
#0 No
#-3 Prefer not to answer

In [ ]:
# Past tobacco smoking - merged never-smoked categories to handle 0s
past_smoking_col = 'Past tobacco smoking-2.0'

past_smoking_mapping = {
    0: 'Never smoked',
    1: 'Smoked on most or all days',
    2: 'Smoked occasionally',
    3: 'Just tried once or twice',
    4: 'Never smoked',
    -3: 'Prefer not to answer'
}

valid_values = set(past_smoking_mapping.keys())

# Validation check
def is_valid_value(x):
    if pd.isna(x):
        return True  # NA values will be handled separately
    try:
        return float(x) in valid_values
    except (ValueError, TypeError):
        return False

# Create mask for invalid values
invalid_mask = ~smoking_i2_dummy[past_smoking_col].apply(is_valid_value) & smoking_i2_dummy[past_smoking_col].notna()

if invalid_mask.any():
    invalid_values = smoking_i2_dummy.loc[invalid_mask, past_smoking_col].unique()
    raise ValueError(f"Unexpected values in column '{past_smoking_col}': {invalid_values}. Update mapping or clean data.")

# Convert with NA
smoking_i2_dummy[past_smoking_col] = (
    smoking_i2_dummy[past_smoking_col]
    .apply(pd.to_numeric, errors='coerce')
    .map(past_smoking_mapping)
)

# One-hot encoding
dummies = pd.get_dummies(
    smoking_i2_dummy[past_smoking_col],
    prefix='Past tobacco smoking'
).astype(int)

# Clean column names
dummies.columns = [col.replace("_", " - ") for col in dummies.columns]

# Final processing
smoking_i2_dummy = pd.concat([smoking_i2_dummy, dummies], axis=1)
smoking_i2_dummy = smoking_i2_dummy.drop(columns=past_smoking_col)

print('SHAPE:', smoking_i2_dummy.shape)
print('NAs:', smoking_i2_dummy.isna().sum().sort_values(ascending=False))

In [ ]:
# Type of tobacco currently smoked
tobacco_type_col = 'Type of tobacco currently smoked-2.0'

tobacco_type_mapping = {
    0: "Don't smoke tobacco",
    1: 'Manufactured cigarettes',
    2: 'Hand-rolled cigarettes',
    3: 'Cigars or pipes',
    -7: 'Other type',
    -3: 'Prefer not to answer'
}

col_numeric = pd.to_numeric(smoking_i2_dummy[tobacco_type_col], errors='coerce')
invalid_mask = ~col_numeric.isin(valid_values) & smoking_i2_dummy[tobacco_type_col].notna()

if invalid_mask.any():
    invalid_values = smoking_i2_dummy.loc[invalid_mask, tobacco_type_col].unique()
    raise ValueError(f"Unexpected values in column '{tobacco_type_col}': {invalid_values}. Update mapping or clean data.")

# Convert with NA protection and mapping
smoking_i2_dummy[tobacco_type_col] = (
    smoking_i2_dummy[tobacco_type_col]
    .apply(pd.to_numeric, errors='coerce')
    .map(tobacco_type_mapping)
)

# One-hot encoding
dummies = pd.get_dummies(
    smoking_i2_dummy[tobacco_type_col],
    prefix='Type of tobacco currently smoked'
).astype(int)

# Clean column names
dummies.columns = [col.replace("_", " - ") for col in dummies.columns]

# Final processing
smoking_i2_dummy = pd.concat([smoking_i2_dummy, dummies], axis=1)
smoking_i2_dummy = smoking_i2_dummy.drop(columns=tobacco_type_col)

print(f"Processed column: {tobacco_type_col}")
print('Current shape:', smoking_i2_dummy.shape)

In [ ]:
# Ever tried to stop smoking
ever_stopped_col = 'Ever tried to stop smoking-2.0'

ever_stopped_mapping = {
    1: 'Yes, tried but was not able to stop or stopped for less than 6 months',
    2: 'Yes, tried and stopped for at least 6 months',
    0: 'No',
    -3: 'Prefer not to answer'
}

valid_values = set(ever_stopped_mapping.keys())

# Check for unexpected values
invalid_mask = (~smoking_i2_dummy[ever_stopped_col].astype(float).isin(valid_values) & 
                smoking_i2_dummy[ever_stopped_col].notna())

if invalid_mask.any():
    invalid_values = smoking_i2_dummy.loc[invalid_mask, ever_stopped_col].unique()
    raise ValueError(f"Unexpected values in '{ever_stopped_col}': {invalid_values}. Update mapping or clean data.")

# Convert
smoking_i2_dummy[ever_stopped_col] = (
    smoking_i2_dummy[ever_stopped_col]
    .astype(float)
    .map(ever_stopped_mapping)
)

# One-hot encoding with all possible categories
dummies = pd.get_dummies(
    smoking_i2_dummy[ever_stopped_col],
    prefix='Ever tried to stop smoking'
).astype(int)

# Clean column names
dummies.columns = [col.replace("_", " - ") for col in dummies.columns]

# Final processing
smoking_i2_dummy = pd.concat([smoking_i2_dummy, dummies], axis=1)
smoking_i2_dummy = smoking_i2_dummy.drop(columns=ever_stopped_col)

# Verification
print(f"\nProcessed column: {ever_stopped_col}")
print('Current shape:', smoking_i2_dummy.shape)
print('Dummy columns created:')
print(dummies.columns.tolist())

In [ ]:
# Wants to stop smoking
wants_stop_mapping = {
    0: "Don't smoke",
    1: 'Yes, definitely',
    2: 'Yes, probably',
    3: 'No, probably not',
    4: 'No, definitely not',
    -3: 'Prefer not to answer'
}

# Process the column in one line
smoking_i2_dummy = process_categorical_column(
    df=smoking_i2_dummy,
    col_name='Wants to stop smoking-2.0',
    mapping=wants_stop_mapping,
    prefix='Wants to stop smoking'
)

# Verification
print(f"\nProcessed column: Wants to stop smoking-2.0")
print('Current shape:', smoking_i2_dummy.shape)

In [ ]:
# Smoking compared to 10 years previous
ten_years_mapping = {
0: "Currently don't smoke",
1:	'More nowadays',
2:	'About the same',
3:	'Less nowadays',
-3:	'Prefer not to answer'
}

smoking_i2_dummy = process_categorical_column(
    df=smoking_i2_dummy,
    col_name='Smoking compared to 10 years previous-2.0',
    mapping=ten_years_mapping,
    prefix='Smoking compared to 10 years previous')

In [ ]:
# Type of tobacco previously smoked

tobacco_type_mapping = {
0: "Didn't smoke tobacco previously",
1:'Manufactured cigarettes',
2:'Hand-rolled cigarettes',
3:'Cigars or pipes',
-7:'Other type',
-3:'Prefer not to answer'}

smoking_i2_dummy = process_categorical_column(
    df=smoking_i2_dummy,
    col_name='Type of tobacco previously smoked-2.0',
    mapping=tobacco_type_mapping,
    prefix='Type of tobacco previously smoked')

In [ ]:
# Likelihood of resuming smoking

likelihood_stop_mapping = {
0: "Don't smoke",
1:'Yes, definitely',
2:'Yes, probably',
3:'No, probably not',
4:'No, definitely not',
-1:'Do not know / Prefer not to answer',
-3:'Prefer not to answer'
}
smoking_i2_dummy = process_categorical_column(
    df=smoking_i2_dummy,
    col_name='Likelihood of resuming smoking-2.0',
    mapping=likelihood_stop_mapping,
    prefix='Likelihood of resuming smoking')

In [ ]:
# Count negative values
print('Number of -1 responses\n', (smoking_i2_dummy == -1).sum().sort_values(ascending=False))
print('Number of -2 responses\n', (smoking_i2_dummy == -2).sum().sort_values(ascending=False))
print('Number of -3 responses\n', (smoking_i2_dummy == -3).sum().sort_values(ascending=False))
print('Number of -7 responses\n', (smoking_i2_dummy == -7).sum().sort_values(ascending=False))
print('Number of -10 responses\n', (smoking_i2_dummy == -10).sum().sort_values(ascending=False))

In [ ]:
smoking_i2_dummy.columns = smoking_i2_dummy.columns.str.replace('-2.0', '')
smoking_i2_dummy.columns.to_list()

In [ ]:
# count NAs
with pd.option_context('display.max_rows', None):
    display(smoking_i2_dummy.isna().sum().sort_values(ascending=False))

In [ ]:
smoking_i2_dummy.replace(-1, np.nan, inplace=True)
smoking_i2_dummy.replace(-3, np.nan, inplace=True)

with pd.option_context('display.max_rows', None):
    display(((smoking_i2_dummy.isna().sum() / len(smoking_i2_dummy)).round(2)*100).sort_values(ascending=False))
    
print('MIN\n', smoking_i2_dummy.min())
print('NEG\n', (smoking_i2_dummy < 0).sum().sort_values(ascending=False))
smoking_i2_dummy.to_csv(os.path.join(data_path, 'smoking.csv'), index=False)

In [ ]:
# Check if there are columns with constant values (e.g., all 0)
def is_constant(column):
    """Check if all values in a column are the same."""
    return np.all(column == column.iloc[0])

def find_constant_columns(df, df_name):
    constant_columns = []
    
    print(f"\nChecking for constant columns in {df_name}...")
    
    for column in df.columns:
        if is_constant(df[column]):
            constant_columns.append(column)
            print(f"✅ Column '{column}' is constant.")
        else:
            print(f"❌ Column '{column}' is NOT constant.")
    
    if constant_columns:
        print(f"\n🔍 Found {len(constant_columns)} constant column(s) in {df_name}: {constant_columns}")
    else:
        print(f"\n🎉 No constant columns found in {df_name}!")
    
    return constant_columns

find_constant_columns(smoking_i2_dummy, "smoking_i2_dummy")

In [ ]:
# Match features to the target and compute sample sizes
folds = range(0,5)
modalities = ['smoking']

modality_observations = {modality: {'train': 0, 'test': 0} for modality in modalities}

for modality in modalities:
    for fold in folds:
        base_path = f'/UK_BB/brainbody/lifestyle'
        folds_path = os.path.join(base_path, f'folds/fold_{fold}')
        suppl_path = os.path.join(folds_path, 'suppl')
        scaling_path = os.path.join(folds_path, 'scaling')
        models_path = os.path.join(folds_path, 'models')
        g_pred_path = os.path.join(folds_path, 'g_pred')
        
        os.makedirs(folds_path, exist_ok=True)
        os.makedirs(suppl_path, exist_ok=True)
        os.makedirs(scaling_path, exist_ok=True)
        os.makedirs(models_path, exist_ok=True)
        os.makedirs(g_pred_path, exist_ok=True)

        g_train = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_train_with_id_{fold}.csv')
        g_test = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_test_with_id_{fold}.csv')
        features = pd.read_csv(os.path.join(base_path, f'data/{modality}.csv'))
        print('Features shape', features.shape)

        feature_columns = features.drop(columns='eid').columns

        # Merge features with targets
        train_merge_all = pd.merge(features, g_train, on='eid').reset_index(drop=True)
        test_merge_all = pd.merge(features, g_test, on='eid').reset_index(drop=True)

        # Save merged data
        train_merge_all.to_csv(os.path.join(suppl_path, f'{modality}_train_feat_targ_fold_{fold}.csv'), index=False)
        test_merge_all.to_csv(os.path.join(suppl_path, f'{modality}_test_feat_targ_fold_{fold}.csv'), index=False)

        print(f'==== Train shape ====\n {modality} - Fold {fold}', train_merge_all.shape)
        print(f'==== Test shape ====\n {modality} - Fold {fold}', test_merge_all.shape)

        # Update the number of observations for the current modality
        modality_observations[modality]['train'] += train_merge_all.shape[0]
        modality_observations[modality]['test'] += test_merge_all.shape[0]

        # Extract features and save
        train_merge_all[feature_columns].to_csv(os.path.join(suppl_path, f'{modality}_train_fold_{fold}.csv'), index=False)
        test_merge_all[feature_columns].to_csv(os.path.join(suppl_path, f'{modality}_test_fold_{fold}.csv'), index=False)

        # Scale features
        print('Scaling')
        scaler_features = StandardScaler()
        features_train_scaled = scaler_features.fit_transform(train_merge_all[feature_columns])
        features_test_scaled = scaler_features.transform(test_merge_all[feature_columns])
        pd.DataFrame(features_train_scaled, columns=feature_columns).to_csv(os.path.join(scaling_path, f'{modality}_train_scaled_fold_{fold}.csv'), index=False)
        pd.DataFrame(features_test_scaled, columns=feature_columns).to_csv(os.path.join(scaling_path, f'{modality}_test_scaled_fold_{fold}.csv'), index=False)

        # Save scaler
        with open(os.path.join(scaling_path, f'{modality}_scaler_features_fold_{fold}.pkl'), "wb") as f:
            pickle.dump(scaler_features, f)

best_modality = max(modality_observations, key=lambda x: modality_observations[x]['train'] + modality_observations[x]['test'])
total_observations = modality_observations[best_modality]['train'] + modality_observations[best_modality]['test']

print('======================================================================================================')
print(f'The modality with the highest number of observations is: {best_modality}: n = {total_observations}')
print(f'Observations in each modality train/test sets:\n', modality_observations)

# Diet: Touchscreen

In [ ]:
# Create directories & save
base_path = f'/UK_BB/brainbody'
lifestyle_path = os.path.join(base_path, 'lifestyle')
data_path = os.path.join(lifestyle_path, 'data')
os.makedirs(base_path, exist_ok=True)
os.makedirs(lifestyle_path, exist_ok=True)
os.makedirs(data_path, exist_ok=True)

In [ ]:
df_diet = ukbiobank.utils.utils.loadCsv(ukbio=ukb, fields=['eid',
1289, #Cooked vegetable intake
1299, #Salad / raw vegetable intake
1309, #Fresh fruit intake
1319, #Dried fruit intake
1329, #Oily fish intake
1339, #Non-oily fish intake
1349, #Processed meat intake
1359, #Poultry intake
1369, #Beef intake
1379, #Lamb/mutton intake
1389, #Pork intake
3680, #Age when last ate meat
6144, #Never eat eggs, dairy, wheat, sugar
1408, #Cheese intake
1418, #Milk type used
1428, #Spread type
2654, #Non-butter spread type details
1438, #Bread intake
1448, #Bread type
1458, #Cereal intake
1468, #Cereal type
1478, #Salt added to food
1488, #Tea intake
1498, #Coffee intake
1508, #Coffee type
1518, #Hot drink temperature
1528, #Water intake
1538, #Major dietary changes in the last 5 years
1548, #Variation in diet
])
df_diet_names = ukbiobank.utils.utils.fieldIdsToNames(ukbio=ukb, df=df_diet)
df_diet_names.to_csv(os.path.join(data_path, 'diet_touchscreen_vars.csv'), index=False)

In [ ]:
print('% missing')
with pd.option_context('display.max_rows', None):
    display(((df_diet_names.isna().sum() / len(df_diet_names)).round(2)*100).sort_values(ascending=False))

In [ ]:
# Filter Instance 2
df_diet_names = pd.read_csv(os.path.join(data_path, 'diet_touchscreen_vars.csv'))
diet_i2 = df_diet_names[['eid'] + df_diet_names.filter(regex=r'2\.\d$').columns.tolist()]
diet_i2.columns = diet_i2.columns.str.replace('-2.0', '')
diet_i2.to_csv(os.path.join(data_path, 'diet_i2.csv'), index=False)
with pd.option_context('display.max_rows', None):
    display(((diet_i2.isna().sum() / len(diet_i2)).round(2)*100).sort_values(ascending=False))

with pd.option_context('display.max_rows', None):
    display(diet_i2.isna().sum())

In [ ]:
# Drop columns with 100% missing values and rename columns
#diet_i2 = diet_i2.drop(columns=[
#'Never eat eggs, dairy, wheat, sugar-2.3',
#'Never eat eggs, dairy, wheat, sugar-2.2',
#'Never eat eggs, dairy, wheat, sugar-2.1',
#'Age when last ate meat'])

diet_i2 = diet_i2.rename(columns={
    'Poultry intake (Field ID: 1359)': 'Poultry intake',
    'Dried fruit intake (Field ID: 1319)': 'Dried fruit intake',
    'Oily fish intake (Field ID: 1329)': 'Oily fish intake',
    'Beef intake (Field ID: 1369)': 'Beef intake',
    'Pork intake (Field ID: 1389)': 'Pork intake'})

First, drop missing values in the subset of the 'head' questions (ie those that did not depend on other questions), i.e. all except:
- Non-butter spread type details
- Cereal type
- Coffee type
- Bread type

In [ ]:
# Drop NAs
diet_i2 = diet_i2.dropna(subset=[
'Cooked vegetable intake',
'Salad / raw vegetable intake',
'Fresh fruit intake',
'Dried fruit intake',
'Oily fish intake',
'Non-oily fish intake',
'Processed meat intake',
'Poultry intake',
'Beef intake',
'Lamb/mutton intake',
'Pork intake',
'Never eat eggs, dairy, wheat, sugar',
'Milk type used',
'Spread type',
'Bread intake',
'Cereal intake',
'Salt added to food',
'Tea intake',
'Coffee intake',
'Hot drink temperature',
'Water intake',
'Major dietary changes in the last 5 years',
'Variation in diet'])

with pd.option_context('display.max_rows', None):
    display(diet_i2.isna().sum())

'''
If 'Never eat eggs, dairy, wheat, sugar' = 'Dairy products', then 'Cheese intake' => 0
If 'Spread type' != 'Other type of spread/margarine' or != 'Do not know', then 'Non-butter spread type details' => 0
If 'Bread intake' = 'Less than one' or 0, then 'Bread type' => 0
If 'Cereal intake' = 'Less than one' or 0, then 'Cereal type' => 0
If 'Coffee intake' != 'Less than one' or >1, then 'Coffee type'  => 0
'''

Next, for qustions that appeared only if the previous question was 'yes' (ie 'head' questions), determine corresponding 'head' questions and fill NA with 0

In [ ]:
# Count NA and negative values
print('Shape', diet_i2.shape)
print('NA\n', (diet_i2.isna()).sum().sort_values(ascending=False))
print('Number of -1 responses\n', (diet_i2 == -1).sum().sort_values(ascending=False))
print('Number of -2 responses\n', (diet_i2 == -2).sum().sort_values(ascending=False))
print('Number of -3 responses\n', (diet_i2 == -3).sum().sort_values(ascending=False))
print('Number of -10 responses\n', (diet_i2 == -10).sum().sort_values(ascending=False))

1. *Cereal type*: collected from all participants except those who indicated that they do not eat cereal or less than one bowl of cereal each week, as defined by their answers to Field 1458 (Cereal intake)
2. *Coffee type*: Field 1508 was collected from participants who indicated that they drank at least 1 cup of coffee each day or less than one cup each day, as defined by their answers to Field 1498 (Coffee intake)
3. *Bread type*: collected from all participants except those who indicated that they do not eat bread or less than one slice each week, as defined by their answers to Field 1438 (Bread intake)
4. *Cheese intake*: collected from all participants except those who indicated that they never eat dairy products, as defined by their answers to Field 6144 (Never eat eggs, dairy, wheat, sugar)
5. *Non-butter spread type details*: collected from participants who indicated they mainly use another type of spread/margarine rather than butter/spreadable butter or do not know what they use, as defined by their answers to Field 1428 (Spread type)

In [ ]:
diet_encode = diet_i2.copy()

In [ ]:
# Fill NA in the dependent questions
# Cereal type
cereal_type = ['Cereal type']
diet_encode.loc[
    diet_encode['Cereal intake'].isin([0, -10]), cereal_type
] = diet_encode.loc[
    diet_encode['Cereal intake'].isin([0, -10]), cereal_type
].fillna(0)

# Coffee type
coffee_type = ['Coffee type']
diet_encode.loc[
    (diet_encode['Coffee intake'] <= 1) & (diet_encode['Coffee intake'] != -10), coffee_type] = diet_encode.loc[
    (diet_encode['Coffee intake'] <= 1) & (diet_encode['Coffee intake'] != -10), coffee_type].fillna(0)

# Bread type
bread_type = ['Bread type']
diet_encode.loc[
    diet_encode['Bread intake'].isin([0, -10]), bread_type
] = diet_encode.loc[
    diet_encode['Bread intake'].isin([0, -10]), bread_type
].fillna(0)

# Cheese intake
cheese_intake = ['Cheese intake']
diet_encode.loc[
    (diet_encode['Never eat eggs, dairy, wheat, sugar'] == 2) | (diet_encode['Never eat eggs, dairy, wheat, sugar-2.1'] == 2) | (diet_encode['Never eat eggs, dairy, wheat, sugar-2.2'] == 2) | (diet_encode['Never eat eggs, dairy, wheat, sugar-2.3'] == 2), cheese_intake
] = diet_encode.loc[
    (diet_encode['Never eat eggs, dairy, wheat, sugar'] == 2) | (diet_encode['Never eat eggs, dairy, wheat, sugar-2.1'] == 2) | (diet_encode['Never eat eggs, dairy, wheat, sugar-2.2'] == 2) | (diet_encode['Never eat eggs, dairy, wheat, sugar-2.3'] == 2), cheese_intake
].fillna(0)

# Non-butter spread type details
non_butter_spread = ['Non-butter spread type details']
diet_encode.loc[
    diet_encode['Spread type'].isin([1, 0, -3, 2]), non_butter_spread
] = diet_encode.loc[
    diet_encode['Spread type'].isin([1, 0, -3, 2]), non_butter_spread
].fillna(0)

# Count NA
print('NA Counts:')
print((diet_encode.isna().sum().sort_values(ascending=False)))

In [ ]:
# Replace -10  with 0.5
'''
diet_encode.loc[:, [
    'Cooked vegetable intake',
    'Salad / raw vegetable intake',
    'Fresh fruit intake',
    'Dried fruit intake',
    'Bread intake',
    'Cereal intake',
    'Tea intake',
    'Coffee intake',
    'Water intake']] = diet_encode.loc[:, [
    'Cooked vegetable intake',
    'Salad / raw vegetable intake',
    'Fresh fruit intake',
    'Dried fruit intake',
    'Bread intake',
    'Cereal intake',
    'Tea intake',
    'Coffee intake',
    'Water intake']].replace({-10: 0.5, #Less than 1
})
'''

diet_encode = diet_encode.replace({-10: 0.5})

print('Number of -10 responses\n', (diet_encode == -10).sum().sort_values(ascending=False))

'Age when last ate meat' has only 27k responses, thus it will be removed

In [ ]:
diet_encode = diet_encode.drop(columns='Age when last ate meat')

#### Dummy-encode non-numeric columns

In [ ]:
# Never eat eggs, dairy, wheat, sugar
never_eat = [
    'Never eat eggs, dairy, wheat, sugar',
    'Never eat eggs, dairy, wheat, sugar-2.1',
    'Never eat eggs, dairy, wheat, sugar-2.2',
    'Never eat eggs, dairy, wheat, sugar-2.3'
]

# Convert to numeric (handling NAs)
diet_encode[never_eat] = diet_encode[never_eat].apply(pd.to_numeric, errors='coerce').astype('Int64')

never_eat_mapping = {
    1: 'Eggs or foods containing eggs',
    2: 'Dairy products',
    3: 'Wheat products',
    4: 'Sugar or foods/drinks containing sugar',
    5: 'I eat all of the above',
    -3: 'Prefer not to answer'
}

# Validation
valid_values = set(never_eat_mapping.keys())
stacked = diet_encode[never_eat].stack()
invalid_mask = ~stacked.isin(valid_values) & stacked.notna()
if invalid_mask.any():
    invalid_values = stacked[invalid_mask].unique()
    raise ValueError(f"Unexpected 'Never eat' codes: {invalid_values}")

# One-hot encoding
dummies = pd.get_dummies(
    stacked[stacked.isin(valid_values)],
    prefix='Never eat'
).groupby(level=0).max()

dummies.columns = [f'Never eat - {never_eat_mapping[int(col.split("_")[-1])]}' 
                  for col in dummies.columns]

# Combine results
diet_encode = pd.concat([diet_encode, dummies], axis=1)
diet_encode.drop(columns=never_eat, inplace=True)

print('Final shape:', diet_encode.shape)
[c for c in diet_encode.columns if 'Never eat - ' in c]

In [ ]:
# Milk type processing
milk_type = 'Milk type used'

# Convert to numeric and handle errors
diet_encode[milk_type] = diet_encode[milk_type].apply(pd.to_numeric, errors='coerce').astype('Int64')

# Define milk type mapping
milk_type_mapping = {
    1: 'Full cream',
    2: 'Semi-skimmed',
    3: 'Skimmed',
    4: 'Soya',
    5: 'Other type of milk',
    6: 'Never/rarely have milk',
    -1: 'Do not know / Prefer not to answer',
    -3: 'Do not know / Prefer not to answer'
}

# Validation
valid_values = set(milk_type_mapping.keys())
stacked = diet_encode[milk_type].dropna()  # Get non-NA values only
invalid_mask = ~stacked.isin(valid_values)

if invalid_mask.any():
    invalid_values = stacked[invalid_mask].unique()
    raise ValueError(f"Unexpected milk type codes: {invalid_values}")

# One-hot encoding
dummies = pd.get_dummies(
    diet_encode[milk_type].map(milk_type_mapping),
    prefix='Milk type used'
).astype(int)

# Clean column names
dummies.columns = [f'Milk type used - {col.split("_")[-1]}' 
                  for col in dummies.columns]

# Combine results
diet_encode = diet_encode.drop(columns=milk_type)
diet_encode = pd.concat([diet_encode, dummies], axis=1)

print('Final shape:', diet_encode.shape)
[c for c in diet_encode.columns if milk_type in c]

In [ ]:
# Spread type
spread_type = 'Spread type'
diet_encode[spread_type] = diet_encode[spread_type].apply(pd.to_numeric, errors='coerce').astype('Int64')
spread_type_mapping = {
    1: 'Butter/spreadable butter',
    3: 'Other type of spread/margarine',
    0: 'Never/rarely use spread',
    -1: 'Do not know / Prefer not to answer',
    -3: 'Do not know / Prefer not to answer',
    2: 'Flora Pro-Active/Benecol'
}

# Validation
valid_values = set(spread_type_mapping.keys())
invalid_values = diet_encode[spread_type][~diet_encode[spread_type].isin(valid_values) & diet_encode[spread_type].notna()]
if not invalid_values.empty:
    raise ValueError(f"Unexpected spread type codes: {invalid_values.unique()}")

# Mapping
diet_encode[spread_type] = diet_encode[spread_type].map(spread_type_mapping)

# One-hot encoding
dummies = pd.get_dummies(
    diet_encode[spread_type], 
    prefix=spread_type).astype(int)

# Clean column name
dummies.columns = [
    f"{spread_type} - {col.split('_')[-1]}"
    for col in dummies.columns
]

# Final concatenation (identical)
diet_encode = diet_encode.drop(columns=spread_type)
diet_encode = pd.concat([diet_encode, dummies], axis=1)

print('Shape', diet_encode.shape)
[c for c in diet_encode.columns if spread_type in c]

In [ ]:
# Non-butter spread type details
non_butter_spread = 'Non-butter spread type details'
diet_encode[non_butter_spread] = diet_encode[non_butter_spread].apply(pd.to_numeric, errors='coerce').astype('Int64')

# Define mapping
non_butter_spread_mapping = {
    4: 'Soft (tub) margarine',
    5: 'Hard (block) margarine',
    6: 'Olive oil based spread (eg: Bertolli)',
    7: 'Polyunsaturated/sunflower oil based spread (eg: Flora)',
    2: 'Flora Pro-Active or Benecol',
    8: 'Other low or reduced fat spread',
    9: 'Other type of spread/margarine',
    -1: 'Do not know / Prefer not to answer',
    -3: 'Do not know / Prefer not to answer',
    0: 'Do not eat spread'
}

# Validation
valid_values = set(non_butter_spread_mapping.keys())
invalid_values = diet_encode[non_butter_spread][
    ~diet_encode[non_butter_spread].isin(valid_values) & 
    diet_encode[non_butter_spread].notna()
]
if not invalid_values.empty:
    raise ValueError(f"Unexpected {non_butter_spread} codes: {invalid_values.unique()}")

# Mapping
diet_encode[non_butter_spread] = diet_encode[non_butter_spread].map(non_butter_spread_mapping)

# One-hot encoding
dummies = pd.get_dummies(
    diet_encode[non_butter_spread],
    prefix=non_butter_spread
).astype(int)

dummies.columns = [
    col.replace("_", " - ")
    for col in dummies.columns
]

# Final output
diet_encode = diet_encode.drop(columns=non_butter_spread)
diet_encode = pd.concat([diet_encode, dummies], axis=1)

print('Shape', diet_encode.shape)
[c for c in diet_encode.columns if non_butter_spread in c]

In [ ]:
# Bread type
bread_type = 'Bread type'
diet_encode[bread_type] = diet_encode[bread_type].apply(pd.to_numeric, errors='coerce').astype('Int64')
bread_type_mapping = {
    1: 'White',
    2: 'Brown',
    3: 'Wholemeal or wholegrain',
    4: 'Other type of bread',
    -1: 'Do not know',
    -3: 'Prefer not to answer',
    0: 'Do not eat bread'
}

valid_values = set(bread_type_mapping.keys())
invalid_values = diet_encode[bread_type][
    ~diet_encode[bread_type].isin(valid_values) & 
    diet_encode[bread_type].notna()
]
if not invalid_values.empty:
    raise ValueError(f"Unexpected {bread_type} codes: {invalid_values.unique()}")

diet_encode[bread_type] = diet_encode[bread_type].map(bread_type_mapping)
dummies = pd.get_dummies(
    diet_encode[bread_type],
    prefix=bread_type
).astype(int)

dummies.columns = [
    col.replace("_", " - ")
    for col in dummies.columns
]

diet_encode = diet_encode.drop(columns=bread_type)
diet_encode = pd.concat([diet_encode, dummies], axis=1)

print('Shape', diet_encode.shape)
[c for c in diet_encode.columns if bread_type in c]

In [ ]:
# Cereal type
cereal_type = 'Cereal type'

# Convert to numeric
diet_encode[cereal_type] = diet_encode[cereal_type].apply(pd.to_numeric, errors='coerce').astype('Int64')

# Define mapping
cereal_type_mapping = {
    1: 'Bran cereal',  # (e.g. All Bran, Branflakes)
    2: 'Biscuit cereal',  # (e.g. Weetabix)
    3: 'Oat cereal',  # (e.g. Ready Brek, porridge)
    4: 'Muesli',
    5: 'Other (e.g. cornflakes, frosties)',
    -1: 'Do not know / Prefer not to answer',
    -3: 'Do not know / Prefer not to answer',
    0: 'Do not eat cereal'  # manual NA handling
}

# Validation
valid_values = set(cereal_type_mapping.keys())
invalid_values = diet_encode[cereal_type][
    ~diet_encode[cereal_type].isin(valid_values) & 
    diet_encode[cereal_type].notna()
]
if not invalid_values.empty:
    raise ValueError(f"Unexpected {cereal_type} codes: {invalid_values.unique()}")

# Mapping
diet_encode[cereal_type] = diet_encode[cereal_type].map(cereal_type_mapping)

# One-hot encoding
dummies = pd.get_dummies(
    diet_encode[cereal_type],
    prefix=cereal_type
).astype(int)

dummies.columns = [
    col.replace("_", " - ")
    for col in dummies.columns
]

# Final output
diet_encode = diet_encode.drop(columns=cereal_type)
diet_encode = pd.concat([diet_encode, dummies], axis=1)

print('Shape', diet_encode.shape)
[c for c in diet_encode.columns if cereal_type in c]

In [ ]:
# Coffee type
coffee_type = 'Coffee type'
diet_encode[coffee_type] = diet_encode[coffee_type].apply(pd.to_numeric, errors='coerce').astype('Int64')
coffee_type_mapping = {
    1: 'Decaffeinated coffee',  # (any type)
    2: 'Instant coffee',
    3: 'Ground coffee (espresso, filter, etc.)',
    4: 'Other type of coffee',
    -1: 'Do not know / Prefer not to answer',
    -3: 'Do not know / Prefer not to answer',
    0: 'Do not drink coffee'  # manual NA handling
}

valid_values = set(coffee_type_mapping.keys())
invalid_values = diet_encode[coffee_type][
    ~diet_encode[coffee_type].isin(valid_values) & 
    diet_encode[coffee_type].notna()
]
if not invalid_values.empty:
    raise ValueError(f"Unexpected {coffee_type} codes: {invalid_values.unique()}")

diet_encode[coffee_type] = diet_encode[coffee_type].map(coffee_type_mapping)
dummies = pd.get_dummies(
    diet_encode[coffee_type],
    prefix=coffee_type
).astype(int)

dummies.columns = [
    col.replace("_", " - ")
    for col in dummies.columns
]

# Final
diet_encode = diet_encode.drop(columns=coffee_type)
diet_encode = pd.concat([diet_encode, dummies], axis=1)

print('Shape', diet_encode.shape)
[c for c in diet_encode.columns if coffee_type in c]

In [ ]:
# Hot drink temperature: recode
diet_encode.loc[:, ['Hot drink temperature']] = diet_encode.loc[:, ['Hot drink temperature']].replace({
1: 3, #'Very hot'
#2: 'Hot'
3: 1, #'Warm'
-2: 0 #'Do not drink hot drinks'
#-3: 'Prefer not to answer'
})
print('Number of -2 responses\n', (diet_encode == -2).sum().sort_values(ascending=False).head(5))

In [ ]:
# Major dietary changes
dietary_changes = 'Major dietary changes in the last 5 years'
diet_encode[dietary_changes] = diet_encode[dietary_changes].apply(pd.to_numeric, errors='coerce').astype('Int64')

# Define mapping
dietary_changes_mapping = {
    0: 'No',
    1: 'Yes, because of illness',
    2: 'Yes, because of other reasons',
    -3: 'Prefer not to answer'
}

# Validation
valid_values = set(dietary_changes_mapping.keys())
invalid_values = diet_encode[dietary_changes][
    ~diet_encode[dietary_changes].isin(valid_values) & 
    diet_encode[dietary_changes].notna()
]
if not invalid_values.empty:
    raise ValueError(f"Unexpected {dietary_changes} codes: {invalid_values.unique()}")

diet_encode[dietary_changes] = diet_encode[dietary_changes].map(dietary_changes_mapping)
dummies = pd.get_dummies(
    diet_encode[dietary_changes],
    prefix=dietary_changes).astype(int)

dummies.columns = [
    col.replace("_", " - ")
    for col in dummies.columns]
diet_encode = diet_encode.drop(columns=dietary_changes)
diet_encode = pd.concat([diet_encode, dummies], axis=1)

print('Shape', diet_encode.shape)
[c for c in diet_encode.columns if dietary_changes in c]

In [ ]:
# Count NA and negative values
print('Shape', diet_encode.shape)
with pd.option_context('display.max_rows', None):
    display(print('NA\n', (diet_encode.isna()).sum().sort_values(ascending=False)))

with pd.option_context('display.max_rows', None):
    display(print('Number of -1 responses\n', (diet_encode == -1).sum().sort_values(ascending=False)))

with pd.option_context('display.max_rows', None):
    display(print('Number of -2 responses\n', (diet_encode == -2).sum().sort_values(ascending=False)))

with pd.option_context('display.max_rows', None):
    display(print('Number of -3 responses\n', (diet_encode == -3).sum().sort_values(ascending=False)))

with pd.option_context('display.max_rows', None):
    display(print('Number of -10 responses\n', (diet_encode == -10).sum().sort_values(ascending=False)))

##### Assign NAs to non-responders

For variables that require dummy-encoding, we will encode non-respondes as well. For numeric/magnitude/frequency responses, non-responders will be encoded as NA not will NOT be removed from the dataset.

In [ ]:
diet_encode.replace(-1, np.nan, inplace=True)
diet_encode.replace(-3, np.nan, inplace=True)

print('MIN\n', diet_encode.min())
print('NEG\n', (diet_encode < 0).sum().sort_values(ascending=False))
with pd.option_context('display.max_rows', None):
    display(print('NA\n', (diet_encode.isna()).sum().sort_values(ascending=False)))
diet_encode.to_csv(os.path.join(data_path, 'diet.csv'), index=False)

In [ ]:
# Match features to target and scale
modalities = ['diet']
seed = 10
algorithm = 'RF'
for modality in modalities:
    model_result = {}
    folds = range(0, 5)

    for fold in folds:

        # Define paths
        base_path = f'/UK_BB/brainbody/lifestyle'
        folds_path = os.path.join(base_path, f'folds/fold_{fold}')
        suppl_path = os.path.join(folds_path, 'suppl')
        scaling_path = os.path.join(folds_path, 'scaling')
        models_path = os.path.join(folds_path, 'models')
        g_pred_path = os.path.join(folds_path, 'g_pred')

        # Create directories if they don't exist
        os.makedirs(folds_path, exist_ok=True)
        os.makedirs(suppl_path, exist_ok=True)
        os.makedirs(scaling_path, exist_ok=True)
        os.makedirs(models_path, exist_ok=True)
        os.makedirs(g_pred_path, exist_ok=True)

        g_train = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_train_with_id_{fold}.csv')
        g_test = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_test_with_id_{fold}.csv')
        features = pd.read_csv(os.path.join(base_path, f'data/{modality}.csv'))
        print('Features shape', features.shape)

        feature_columns = features.drop(columns='eid').columns

        # Merge features with targets
        train_merge_all = pd.merge(features, g_train, on='eid').reset_index(drop=True)
        test_merge_all = pd.merge(features, g_test, on='eid').reset_index(drop=True)

        # Save merged data
        train_merge_all.to_csv(os.path.join(suppl_path, f'{modality}_train_feat_targ_fold_{fold}.csv'), index=False)
        test_merge_all.to_csv(os.path.join(suppl_path, f'{modality}_test_feat_targ_fold_{fold}.csv'), index=False)

        # Extract features and save
        features_train = train_merge_all[feature_columns]
        features_test = test_merge_all[feature_columns]
        features_train.to_csv(os.path.join(suppl_path, f'{modality}_train_fold_{fold}.csv'), index=False)
        features_test.to_csv(os.path.join(suppl_path, f'{modality}_test_fold_{fold}.csv'), index=False)
        print('Train shape, eid dropped', features_train.shape)
        print('Test shape, eid dropped', features_test.shape)

        # Extract targets
        g_train = train_merge_all['g']
        g_test = test_merge_all['g']

        # Scale features
        print('Scaling')
        scaler_features = StandardScaler()
        features_train_scaled = scaler_features.fit_transform(features_train)
        features_test_scaled = scaler_features.transform(features_test)

        # Save scaled features
        pd.DataFrame(features_train_scaled, columns=feature_columns).to_csv(os.path.join(scaling_path, f'{modality}_train_scaled.csv'), index=False)
        pd.DataFrame(features_test_scaled, columns=feature_columns).to_csv(os.path.join(scaling_path, f'{modality}_test_scaled.csv'), index=False)

        # Save scaler
        with open(os.path.join(scaling_path, f'{modality}_scaler_features_fold_{fold}.pkl'), "wb") as f:
            pickle.dump(scaler_features, f)

## Local environment

In [ ]:
df_localenv = ukbiobank.utils.utils.loadCsv(ukbio=ukb, fields=['eid', 
24014, #Close to major road
24012, #Inverse distance to the nearest major road
24010, #Inverse distance to the nearest road
24016, #Nitrogen dioxide air pollution; 2005
24017, #Nitrogen dioxide air pollution; 2006
24018, #Nitrogen dioxide air pollution; 2007
24003, #Nitrogen dioxide air pollution; 2010
24004, #Nitrogen oxides air pollution; 2010
24019, #Particulate matter air pollution (pm10); 2007
24005, #Particulate matter air pollution (pm10); 2010
24007, #Particulate matter air pollution (pm2.5) absorbance; 2010
24006, #Particulate matter air pollution (pm2.5); 2010
24008, #Particulate matter air pollution 2.5-10um; 2010
24015, #Sum of road length of major roads within 100m
24013, #Total traffic load on major roads
24011, #Traffic intensity on the nearest major road
24009, #Traffic intensity on the nearest road
24023, #Average 16-hour sound level of noise pollution
24024, #Average 24-hour sound level of noise pollution
24020, #Average daytime sound level of noise pollution
24021, #Average evening sound level of noise pollution
24022, #Average night-time sound level of noise pollution
24506, #Natural environment percentage, buffer 1000m
24507, #Natural environment percentage, buffer 300m
24500, #Greenspace percentage, buffer 1000m
24503, #Greenspace percentage, buffer 300m
24501, #Domestic garden percentage, buffer 1000m
24504, #Domestic garden percentage, buffer 300m
24502, #Water percentage, buffer 1000m
24505, #Water percentage, buffer 300m
24508, #Distance (Euclidean) to coast
21104, #Ca concentration
21103, #CaCO3 concentration
21105, #Mg concentration
21100, #Water hardness (USGS classification)
21101, #Water hardness (WHO classification)
#21102 #Year of survey
])
df_localenv_names = ukbiobank.utils.utils.fieldIdsToNames(ukbio=ukb, df=df_localenv)
df_localenv_names.to_csv('/UK_BB/brainbody/envir/data/localenv_vars.csv', index=False)

In [ ]:
# Get water minerals and merge them to other features
water_minerals = pd.read_csv('/UK_BB/ukbbdataukb/ukb.csv', usecols=lambda col: col == 'eid' or col.startswith('21100-') or col.startswith('21101-') or col.startswith('21103-') or col.startswith('21104-') or col.startswith('21105-'))

# Not available
'21104-0.0' #Ca concentration
'21103-0.0' #CaCO3 concentration
'21105-0.0' #Mg concentration 

#lambda function is used to create an anonymous function that checks each column name in the CSV file
#lambda col: defines an anonymous function that takes one argument, col, which represents a column name, then checks if col is equal to 'eid' or if it startswith()

water_minerals.to_csv('/UK_BB/brainbody/envir/data/water_minerals.csv', index=False)
water_minerals = pd.read_csv('/UK_BB/brainbody/envir/data/water_minerals.csv')

water_minerals = water_minerals[['eid', '21100-0.0', '21101-0.0']]

water_minerals.columns = water_minerals.columns.str.replace('21100-0.0', 'Water hardness (USGS classification)-0.0')
water_minerals.columns = water_minerals.columns.str.replace('21101-0.0', 'Water hardness (WHO classification)-0.0')

# Merge data
localenv = df_localenv_names.merge(water_minerals, on = 'eid')
# Save
localenv.to_csv('/UK_BB/brainbody/envir/data/localenv_raw.csv', index=False)

In [ ]:
# Filter Instance 0
localenv_i0 = localenv[['eid'] + localenv.filter(regex=r'0\.\d$').columns.tolist()]
localenv_i0.columns = localenv_i0.columns.str.replace('-0.0', '')
# Calculate proportion of missing values
print('% missing')
((localenv_i0.isna().sum() / len(localenv_i0)).round(2)*100).sort_values(ascending=False)

In [ ]:
# Drop NA & save
localenv_i0 = localenv_i0.dropna(axis=0).reset_index(drop=True)
localenv_i0.to_csv('/UK_BB/brainbody/envir/data/localenv_vars_i0_nona.csv', index=False)
localenv_i0.to_csv('/UK_BB/brainbody/envir/data/localenv.csv', index=False)

#Display max, min, and negative values
print('Shape', localenv_i0.shape)
print('MIN\n', localenv_i0.min().round(3))
print('MAX\n', localenv_i0.max())
print('NEG\n', (localenv_i0 < 0).sum().sort_values(ascending=False))
print('NA\n', (localenv_i0.isna()).sum().sort_values(ascending=False))

In [ ]:
# Match features and targets and scale
warnings.simplefilter(action='ignore', category=FutureWarning)
modalities = ['localenv']

for modality in modalities:
    model_result = {}
    folds = range(0, 5)

    for fold in folds:
        
        g_train = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_train_with_id_{fold}.csv')
        g_test = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_test_with_id_{fold}.csv')
        features = pd.read_csv(f'/UK_BB/brainbody/envir/data/{modality}.csv')
        print('Features shape', features.shape)

        feature_columns = features.drop(columns='eid').columns

        train_merge_all = pd.merge(features, g_train, on = 'eid').reset_index(drop=True)
        train_merge_all.to_csv(f'/UK_BB/brainbody/envir/folds/fold_{fold}/suppl/{modality}_train_feat_targ_fold_{fold}.csv', index=False)

        test_merge_all = pd.merge(features, g_test, on = 'eid').reset_index(drop=True)
        train_merge_all.to_csv(f'/UK_BB/brainbody/envir/folds/fold_{fold}/suppl/{modality}_test_feat_targ_fold_{fold}.csv', index=False)

        features_train = train_merge_all[feature_columns]
        features_train.to_csv(f'/UK_BB/brainbody/envir/folds/fold_{fold}/suppl/{modality}_train_fold_{fold}.csv', index=False)
        print('Train shape, eid dropped', features_train.shape)

        features_test = test_merge_all[feature_columns]
        features_test.to_csv(f'/UK_BB/brainbody/envir/folds/fold_{fold}/suppl/{modality}_test_fold_{fold}.csv', index=False)
        print('Test shape, eid dropped', features_test.shape)

        g_train = train_merge_all['g']
        g_test = test_merge_all['g']

        # Scale
        print('Scaling')
        scaler_features = StandardScaler()
        features_train_scaled, features_test_scaled = scaler_features.fit_transform(features_train), scaler_features.transform(features_test)

        pd.DataFrame(features_train_scaled, columns = feature_columns).to_csv(f'/UK_BB/brainbody/envir/folds/fold_{fold}/scaling/{modality}_train_scaled.csv', index=False)
        pd.DataFrame(features_test_scaled, columns = feature_columns).to_csv(f'/UK_BB/brainbody/envir/folds/fold_{fold}/scaling/{modality}_test_scaled.csv', index=False)

        
        with open(f'/UK_BB/brainbody/envir/folds/fold_{fold}/scaling/{modality}_scaler_features_fold_{fold}.pkl', "wb") as f:
            pickle.dump(scaler_features, f)

# Sleep

In [ ]:
df_sleep_ac = ukbiobank.utils.utils.loadCsv(ukbio=ukb, fields=['eid',
#Assessment centre ⏵ Touchscreen ⏵ Lifestyle and environment ⏵ Sleep
1160, #Sleep duration
1170, #Getting up in morning
1180, #Morning/evening person (chronotype)
1190, #Nap during day
1200, #Sleeplessness / insomnia
1210, #Snoring
1220 #Daytime dozing / sleeping
])
df_sleep_ac_names = ukbiobank.utils.utils.fieldIdsToNames(ukbio=ukb, df=df_sleep_ac)
df_sleep_ac_names.to_csv('/UK_BB/brainbody/sleep/data/sleep_ac_vars.csv', index=False)

In [ ]:
print('% missing')
with pd.option_context('display.max_rows', None):
    display(((df_sleep_ac_names.isna().sum() / len(df_sleep_ac_names)).round(2)*100).sort_values(ascending=False))

In [ ]:
# Filter Instances 0 and 2
sleep_ac_i0_i2 = df_sleep_ac_names[['eid'] + df_sleep_ac_names.filter(regex=r'0\.\d$|2\.\d$').columns.tolist()]
sleep_ac_i0_i2.to_csv('/UK_BB/brainbody/sleep/data/sleep_ac_i0_i2.csv', index=False)
with pd.option_context('display.max_rows', None):
    display(((sleep_ac_i0_i2.isna().sum() / len(sleep_ac_i0_i2)).round(2)*100).sort_values(ascending=False))

In [ ]:
# Filter Instances 3 and 4
sleep_ac_i3_i4 = df_sleep_ac_names[['eid'] + df_sleep_ac_names.filter(regex=r'3\.\d$|4\.\d$').columns.tolist()]
sleep_ac_i3_i4.to_csv('/UK_BB/brainbody/sleep/data/sleep_ac_i3_i4.csv', index=False)
with pd.option_context('display.max_rows', None):
    display(((sleep_ac_i3_i4.isna().sum() / len(sleep_ac_i3_i4)).round(2)*100).sort_values(ascending=False))

### Extract and use Instance 2

In [ ]:
# Filter Instance 2
sleep_ac_i2 = sleep_ac_i0_i2[['eid'] + df_sleep_ac_names.filter(regex=r'2\.\d$').columns.tolist()]
sleep_ac_i2.columns = sleep_ac_i2.columns.str.replace('-2.0', '')
sleep_ac_i2.to_csv('/UK_BB/brainbody/sleep/data/sleep_ac_i2.csv', index=False)

In [ ]:
# Count missing values, negative values, min, and max
print('% missing\n', ((sleep_ac_i2.isna().sum() / len(sleep_ac_i2)).round(2)*100).sort_values(ascending=False))
print('MIN\n', sleep_ac_i2.min())
print('MAX\n', sleep_ac_i2.max())
print('NEG\n', (sleep_ac_i2 < 0).sum().sort_values(ascending=False))

In [ ]:
print('Number of -1 responses\n', (sleep_ac_i2 == -1).sum().sort_values(ascending=False))
print('Number of -3 responses\n', (sleep_ac_i2 == -3).sum().sort_values(ascending=False))
print('------------------------------------------------------------------------------------------------------------------------')
print('Proportion of -1 responses\n', (((sleep_ac_i2 == -1).sum() / len(sleep_ac_i2)).round(4)*100).sort_values(ascending=False))
print('Proportion of -3 responses\n', (((sleep_ac_i2 == -3).sum() / len(sleep_ac_i2)).round(4)*100).sort_values(ascending=False))

#### Data encoding

##### Dummy-encode responses in 'Morning/evening person (chronotype)'

In [ ]:
sleep_ac_i2_encoding = sleep_ac_i2.copy()

In [ ]:
# Morning/evening person (chronotype)
chronotype = ['Morning/evening person (chronotype)']
sleep_ac_i2_encoding[chronotype] = (
    sleep_ac_i2_encoding[chronotype]
    .apply(pd.to_numeric, errors='coerce')
    .astype('Int64')
)
# Define mapping (with underscores for safer column names)
chronotype_mapping = {
1:"Definitely a 'morning' person",
2:"More a 'morning' than 'evening' person",
3:"More an 'evening' than a 'morning' person",
4:"Definitely an 'evening' person",
-1:"Do not know / Prefer not to answer",
-3:"Do not know / Prefer not to answer"
}

# One-hot encoding with explicit NA handling
stacked = sleep_ac_i2_encoding[chronotype].stack()
valid_values = stacked.isin(chronotype_mapping.keys())

dummies = (
    pd.get_dummies(
        stacked[valid_values].map(chronotype_mapping),
        prefix='Chronotype',
        prefix_sep=' - '
    )
    .groupby(level=0)
    .sum()
    .astype('Int64')
)

# Ensure all categories are represented
for category in chronotype_mapping.values():
    col_name = f"Chronotype - {category}"
    if col_name not in dummies:
        dummies[col_name] = 0

# Merge and clean
sleep_ac_i2_encoding = pd.concat([
    sleep_ac_i2_encoding.drop(columns=chronotype),
    dummies
], axis=1)


print('Final shape:', sleep_ac_i2_encoding.shape)

##### Change response encoding in 'Snoring'

In [ ]:
''' Original encoding 
1 Yes
2 No
-1 Do not know
-3 Prefer not to answer
'''

'''Change to
1 Yes
0 No
-1 Do not know
-3 Prefer not to answer
'''

In [ ]:
# Replace 2 with 0 in the 'Snoring' column
sleep_ac_i2_encoding['Snoring'] = sleep_ac_i2_encoding['Snoring'].replace(2, 0)

In [ ]:
print('MIN\n', sleep_ac_i2_encoding.min())
print('MAX\n', sleep_ac_i2_encoding.max())
print('NEG\n', (sleep_ac_i2_encoding < 0).sum().sort_values(ascending=False))

Drop missing values

In [ ]:
# Drop NAs
sleep_ac_i2_encoding = sleep_ac_i2_encoding.dropna(axis=0).reset_index(drop=True)
print('Shape', sleep_ac_i2_encoding.shape)
print('% missing\n', ((sleep_ac_i2_encoding.isna().sum() / len(sleep_ac_i2_encoding)).round(2)*100).sort_values(ascending=False))

In [ ]:
print('Number of -1 responses\n', (sleep_ac_i2_encoding == -1).sum().sort_values(ascending=False))
print('Number of -3 responses\n', (sleep_ac_i2_encoding == -3).sum().sort_values(ascending=False))
print('------------------------------------------------------------------------------------------------------------------------')
print('Proportion of -1 responses\n', (((sleep_ac_i2_encoding == -1).sum() / len(sleep_ac_i2_encoding)).round(4)*100).sort_values(ascending=False))
print('Proportion of -3 responses\n', (((sleep_ac_i2_encoding == -3).sum() / len(sleep_ac_i2_encoding)).round(4)*100).sort_values(ascending=False))

________________________________________________________________________________________________________
##### Non-responders: 'Do not know' (-1) and 'Prefer not to answer' (-3)

1. [Patterns of item nonresponse behaviour to survey questionnaires are systematic and associated with genetic loci](https://pmc.ncbi.nlm.nih.gov/articles/PMC10444625/)
2. [Dietary assessment in UK Biobank: an evaluation of the performance of the touchscreen dietary questionnaire](https://pmc.ncbi.nlm.nih.gov/articles/PMC5799609/)

'For all questions, participants selecting ‘do not know’, ‘prefer not to answer’ or ‘less than one’ were assigned to separate categories, except for number of bread slices where ‘less than one’ was combined with ‘0’ because of very low numbers for both of these groups.'

3. [Comparison of Sociodemographic and Health-Related Characteristics of UK Biobank Participants With Those of the General Population](https://academic.oup.com/aje/article/186/9/1026/3883629)

'Excludes 2,778 UK Biobank participants aged 40–69 years who were missing data on ethnicity or responded “prefer not to answer” or “do not know.'

4. [Replacement of sedentary behavior with various physical activities and the risk of all-cause and cause-specific mortality](https://pmc.ncbi.nlm.nih.gov/articles/PMC11395964/#Sec2)

'For categorical variables, systematic missing values and responses of “do not know” or “prefer not to answer” were consolidated into the “missing” category.'
________________________________________________________________________________________________________

##### 1. Assign NAs to non-responders

In [ ]:
sleep_ac_i2_encoding.replace(-1, np.nan, inplace=True)
sleep_ac_i2_encoding.replace(-3, np.nan, inplace=True)

print('MIN\n', sleep_ac_i2_encoding.min())
print('NEG\n', (sleep_ac_i2_encoding < 0).sum().sort_values(ascending=False))
print('NA\n', (sleep_ac_i2_encoding.isna()).sum().sort_values(ascending=False))

sleep_ac_i2_encoding.to_csv('/UK_BB/brainbody/sleep/data/sleep_ac_vars_i2_na.csv', index=False)
sleep_ac_i2_encoding.to_csv('/UK_BB/brainbody/sleep/data/sleep_ac.csv', index=False)
print('NA\n', (pd.read_csv('/UK_BB/brainbody/sleep/data/sleep_ac.csv').isna()).sum().sort_values(ascending=False))

##### 2. Drop NAs from non-responders

In [ ]:
sleep_ac_i2_nona = sleep_ac_i2_encoding.dropna(axis=0).reset_index(drop=True)

print('MIN\n', sleep_ac_i2_nona.min())
print('NEG\n', (sleep_ac_i2_nona < 0).sum().sort_values(ascending=False))
print('NA\n', (sleep_ac_i2_nona.isna()).sum().sort_values(ascending=False))

sleep_ac_i2_nona.to_csv('/UK_BB/brainbody/sleep/data/sleep_ac_vars_i2_nona.csv', index=False)
sleep_ac_i2_nona.to_csv('/UK_BB/brainbody/sleep/data/sleep_ac_nona.csv', index=False)

In [ ]:
# Match features to targets and scale
modalities = ['sleep_ac']
seed = 10

for modality in modalities:
    model_result = {}
    folds = range(0, 5)

    for fold in folds:

        # Define paths
        base_path = f'/UK_BB/brainbody/sleep'
        folds_path = os.path.join(base_path, f'folds/fold_{fold}')
        suppl_path = os.path.join(folds_path, 'suppl')
        scaling_path = os.path.join(folds_path, 'scaling')
        models_path = os.path.join(folds_path, 'models')
        g_pred_path = os.path.join(folds_path, 'g_pred')

        # Create directories if they don't exist
        os.makedirs(folds_path, exist_ok=True)
        os.makedirs(suppl_path, exist_ok=True)
        os.makedirs(scaling_path, exist_ok=True)
        os.makedirs(models_path, exist_ok=True)
        os.makedirs(g_pred_path, exist_ok=True)


        g_train = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_train_with_id_{fold}.csv')
        g_test = pd.read_csv(f'/UK_BB/brainbody/cognition/folds/fold_{fold}/g/g_test_with_id_{fold}.csv')
        features = pd.read_csv(f'/UK_BB/brainbody/sleep/data/{modality}.csv')
        print('Features shape', features.shape)

        feature_columns = features.drop(columns='eid').columns

        # Merge features with targets
        train_merge_all = pd.merge(features, g_train, on='eid').reset_index(drop=True)
        test_merge_all = pd.merge(features, g_test, on='eid').reset_index(drop=True)

        # Save merged data
        train_merge_all.to_csv(os.path.join(suppl_path, f'{modality}_train_feat_targ_fold_{fold}.csv'), index=False)
        test_merge_all.to_csv(os.path.join(suppl_path, f'{modality}_test_feat_targ_fold_{fold}.csv'), index=False)

        # Extract features and save
        features_train = train_merge_all[feature_columns]
        features_test = test_merge_all[feature_columns]
        features_train.to_csv(os.path.join(suppl_path, f'{modality}_train_fold_{fold}.csv'), index=False)
        features_test.to_csv(os.path.join(suppl_path, f'{modality}_test_fold_{fold}.csv'), index=False)
        print('Train shape, eid dropped', features_train.shape)
        print('Test shape, eid dropped', features_test.shape)

        # Extract targets
        g_train = train_merge_all['g']
        g_test = test_merge_all['g']

        # Scale features
        print('Scaling')
        scaler_features = StandardScaler()
        features_train_scaled = scaler_features.fit_transform(features_train)
        features_test_scaled = scaler_features.transform(features_test)

        # Save scaled features
        pd.DataFrame(features_train_scaled, columns=feature_columns).to_csv(os.path.join(scaling_path, f'{modality}_train_scaled.csv'), index=False)
        pd.DataFrame(features_test_scaled, columns=feature_columns).to_csv(os.path.join(scaling_path, f'{modality}_test_scaled.csv'), index=False)

        # Save scaler
        with open(os.path.join(scaling_path, f'{modality}_scaler_features_fold_{fold}.pkl'), "wb") as f:
            pickle.dump(scaler_features, f)